# 05 · Modelado y evaluación — el banco de pruebas

Este cuaderno es el banco donde se prueban conjuntos de variables y familias de
modelos contra el mercado. No es un cuaderno de lectura pasiva: las celdas de la
**Parte C** están pensadas para editarse y ejecutarse muchas veces, una por cada
conjunto que el analista quiera medir.

## Qué hace

1. Declara el **protocolo** de validación —ventana móvil por jornada, preprocesado
   ajustado dentro de cada pliegue, recencia opcional— y lo audita antes de usarlo.
2. Declara las **cuatro familias** del estudio (logística multinomial, bosque
   aleatorio, potenciación por gradiente y red neuronal) con una configuración fija.
3. Ofrece un **banco de ensayos** (`laboratorio.py`) que, dado un conjunto de
   variables, devuelve sus métricas de clasificación y su rendimiento económico
   frente al apostante de referencia de Bet365.
4. Ejecuta la **búsqueda hacia delante** (`seleccion.py`) sobre un bloque de
   temporadas distinto de aquel del que sale la cifra publicable, de modo que
   haber probado decenas de variables no infle lo que después se cita.
5. Registra cada ensayo en un **libro acumulativo** (`registro.parquet`) para poder
   comparar, recuperar y rehacer cualquier resultado de cualquier sesión.
6. Somete el conjunto elegido a una **lectura completa**: estabilidad temporal,
   calibración, matriz de confusión, desacuerdo con el mercado, curva de capital e
   interpretabilidad por bloques.

## Qué NO hace

- **No elige y publica en el mismo sitio.** La búsqueda de C.8–C.10 recorre las
  candidatas tantas veces como haga falta, pero lo hace sobre las temporadas
  anteriores a `protocol.CORTE_SELECCION`; la cifra que se cita se mide sobre las
  posteriores, con el conjunto ya cerrado. El bloque no es un argumento que se
  pueda cambiar por descuido: pedirlo del revés levanta `PermissionError`.
- **No decide sola.** El criterio de la búsqueda —el RPS— se declara antes de
  buscar, y la elección final, con su familia y sus hiperparámetros, se cierra en
  `build_models.py`, que la recoge de `seleccion.carga()` en vez de por copia.
- **No abre el conjunto de reserva.** La temporada 2025/26 permanece sellada: el
  banco se niega a evaluar sobre ella (`evalua(..., evaluar="holdout")` lanza
  `PermissionError`). Se abre una sola vez, fuera de aquí, con el conjunto ya cerrado.
- **No barre el espacio de hiperparámetros durante la búsqueda de variables.**
  Cada familia lleva una configuración fija declarada; el afinado corto llega al
  final, sobre el conjunto ya elegido.
- **No publica cifras económicas sueltas.** Todo rendimiento se presenta emparejado
  con el del apostante de referencia sobre los mismos partidos.

## El contrato de esta fase

> El banco **mide**; el analista **decide**.

De ahí se derivan cinco compromisos que este cuaderno respeta de principio a fin:

- **Se elige en un sitio y se publica en otro.** Las variables se eligen sobre las
  temporadas anteriores al corte de selección; la cifra citable se mide una sola vez
  sobre las posteriores, con el conjunto ya cerrado.
- **El criterio se enuncia antes de mirar.** La sección A.5 fija qué se va a mirar
  y en qué orden, y se escribe antes de la primera tabla de resultados.
- **Toda cifra económica va emparejada.** El listón es el apostante que sigue al
  favorito de Bet365 sobre los mismos partidos, temporada a temporada.
- **Todo ensayo queda registrado.** Nombre, familia, variables, protocolo, métricas
  y sello criptográfico van al libro acumulativo; nada depende de la memoria de la
  sesión.
- **La reserva se abre una sola vez.** Y nunca desde aquí.

---
# Parte A · El protocolo

Antes de medir nada hay que fijar **cómo** se mide. Esta parte declara el esquema de
validación, lo audita, y enuncia el criterio de lectura.

## A.0 · Entorno y carga

Se cargan la matriz preprocesada de la fase 04 y los módulos del banco. `laboratorio`
es la fachada de trabajo: encapsula protocolo, evaluación y contabilidad económica
detrás de una sola llamada.

In [ ]:
from __future__ import annotations

import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for _p in (RAIZ / "scripts" / "models", RAIZ / "scripts" / "features"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import columns as C
import protocol as pt
import preprocess as pp
import estimators as es
import evaluate as ev
import backtest as bt
import explain as ex
import laboratorio as lb
import seleccion as sl

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
PALETA = {"modelo": "#1f4e79", "mercado": "#c0504d", "apoyo": "#7f7f7f"}

# Todo ensayo exploratorio de este cuaderno se mide sobre el bloque de
# SELECCIÓN, y no sobre el de validación entero. El banco de `laboratorio`
# acepta cualquier bloque menos el holdout y la temporada en curso, de modo que
# el límite no lo pone el código: lo pone el método. Comparar conjuntos a mano
# sobre 2022/23-2024/25 gastaría el bloque que C.10 publica como la única cifra
# citable del trabajo, y lo gastaría sin dar ningún error y devolviendo una
# tabla de aspecto perfectamente normal.
BLOQUE_BANCO = sl.BLOQUE_SELECCION      # "seleccion" = 2019/20-2021/22

feat = lb.datos()
print(f"matriz de variables : {feat.shape[0]:,} partidos x {feat.shape[1]:,} columnas")
print(f"temporadas          : {feat[C.SEASON].min()} -> {feat[C.SEASON].max()}")
print(f"reparto             : {feat[C.SPLIT].value_counts().to_dict()}")
print(f"variables candidatas: {len(lb.columnas_disponibles(feat)):,}")

## A.1 · El régimen de columnas

El **régimen** dice qué información se le permite ver al modelo. Es una decisión de la
ejecución entera —no una etiqueta pegada a cada columna— y sólo hay dos, declarados en
`columns.REGIMES`:

- **`semanal`** — únicamente lo que existe en el momento de emitir la predicción, antes
  de que se juegue la jornada. Es el régimen de uso real y el único bajo el cual una
  cifra económica significa algo, porque es la única apuesta que se habría podido
  colocar de verdad.
- **`retrospectivo`** — todo, incluido el bloque derivado del **precio de cierre**, que
  no se conoce hasta que el mercado cierra y por tanto ya después del momento de
  apostar. Existe para medir cuánto valía ese precio; sus cifras son informativas y no
  emitibles.

El cortafuegos contra la fuga temporal no es una convención de nombres: es una guarda
ejecutable. `protocol._guardas_matriz` aborta el ensayo si una matriz declarada semanal
contiene una sola columna del bloque de cierre. Por eso las candidatas se filtran por el
régimen aquí, antes de componer nada, y no más tarde.


In [ ]:
# El régimen de trabajo. Cámbielo a "retrospectivo" sólo para medir qué aportaba el
# precio de cierre, sabiendo que esa cifra no se puede emitir.
REGIMEN = "semanal"

candidatas = lb.columnas_disponibles(feat)
usables    = C.regime_columns(candidatas, REGIMEN)
excluidas  = sorted(set(candidatas) - set(usables))

display(pd.Series({r: len(C.regime_columns(candidatas, r)) for r in C.REGIMES},
                  name="candidatas").to_frame())

print(f"régimen              : {REGIMEN}")
print(f"candidatas usables   : {len(usables):,}")
print(f"excluidas por régimen: {len(excluidas)}")
for c in excluidas:
    print("   ·", c)


## A.2 · Jornada efectiva y pliegues

El protocolo avanza **por jornada**, no por temporada. Cada pliegue es una jornada
evaluable; su ventana de entrenamiento es todo lo estrictamente anterior por fecha.
Esto reproduce la situación real de uso —predecir la jornada que viene con lo que se
sabía hasta el viernes— y evita el artificio de entrenar con partidos posteriores.

La *jornada efectiva* agrupa los partidos aplazados con los de su ventana natural, de
modo que un partido movido de fecha no rompe el orden temporal.

In [ ]:
inf = pt.matchday_report(feat)
display(inf.head(12))
print(f"jornadas efectivas totales: {len(inf):,}")

In [ ]:
cfg = pt.ProtocolConfig()
print(cfg.describe())

pl = lb.pliegues(cfg, feat=feat)
rep = pt.folds_report(pl)
display(rep.tail(12))
print(f"pliegues evaluables: {len(pl):,}")
print(f"partidos evaluados : {int(rep['n_eval'].sum()):,}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax[0].plot(rep.index, rep["n_train"], color=PALETA["modelo"], lw=1.6)
ax[0].set_ylabel("partidos de entrenamiento")
ax[0].set_title("Crecimiento de la ventana de entrenamiento, pliegue a pliegue")

ax[1].bar(rep.index, rep["n_eval"], color=PALETA["apoyo"], width=1.0)
ax[1].set_ylabel("partidos evaluados")
ax[1].set_xlabel("pliegue (orden cronológico)")
ax[1].set_title("Tamaño de cada jornada evaluada")

plt.tight_layout(); plt.show()

In [ ]:
solape = [i for i, f in enumerate(pl)
          if len(set(f.train_idx) & set(f.eval_idx)) > 0]
assert not solape, f"pliegues con solape entrenamiento/prueba: {solape[:5]}"

desorden = [i for i, f in enumerate(pl)
            if feat.loc[f.train_idx, C.DATE].max() >= feat.loc[f.eval_idx, C.DATE].min()]
assert not desorden, f"pliegues con entrenamiento posterior a la prueba: {desorden[:5]}"

print(f"OK · {len(pl):,} pliegues sin solape y en orden estrictamente creciente")

## A.3 · El preprocesado vive dentro del pliegue

Imputación, escalado, transformaciones y descartes por correlación se ajustan **sólo**
con la ventana de entrenamiento de cada pliegue y se aplican a la jornada evaluada.
Ajustar una sola vez sobre toda la matriz sería fuga: la media con la que se imputa un
partido de 2019 no puede contener información de 2024.

El estado resultante depende del protocolo, no del modelo ni de las columnas elegidas.
Por eso el banco lo **cachea** y lo reutiliza entre ensayos: es la razón de que probar
el decimoquinto conjunto cueste una fracción de lo que costó el primero.

In [ ]:
estados = lb.estados(cfg, feat=feat)
print(f"estados de preprocesado ajustados: {len(estados):,}")

deriva = pt.imputation_drift(estados)
display(deriva.head(10))

fig, ax = plt.subplots(figsize=(12, 4))
for col in deriva.columns[:6]:
    ax.plot(deriva.index, deriva[col], lw=1.2, label=col)
ax.set_xlabel("pliegue"); ax.set_ylabel("valor de imputación")
ax.set_title("Deriva de los valores de imputación a lo largo del histórico")
ax.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

## A.4 · La recencia como eje declarado

¿Pesa igual un partido de 2016 que uno de la jornada pasada? El protocolo admite una
**semivida** en días: cada partido de entrenamiento recibe un peso que decae
exponencialmente con su antigüedad. `halflife_days=None` significa peso uniforme.

Esto **no** es un hiperparámetro que el banco optimice por su cuenta: es un eje del
protocolo que el analista puede comparar a mano, igual que compara conjuntos de
variables. `pt.SEMIVIDAS` es la lista de candidatas declarada.

El panel siguiente traduce cada semivida a un **tamaño muestral efectivo** (fórmula de
Kish): cuántos partidos "cuentan" de verdad bajo ese decaimiento. Sirve para ver el
canje —más recencia, menos muestra— antes de decidir nada.
**Y la comparación ya se hizo, con resultado negativo.** Las treinta combinaciones de
ventana (expansiva y deslizante de 2 a 6 temporadas) por semivida (uniforme, 180, 365,
730 y 1460 días), sobre el bloque de selección y con dos conjuntos —la semilla de
mercado y el de cinco variables—, dejan **la expansiva sin peso como la mejor de las
treinta en las dos lecturas**, sin ninguna que la bata. Acortar la ventana sin ponderar
es indiferente (deslizante-5 pierde 0.000028 y 0.000015 de RPS); ponderar empeora de
forma monótona, y la semivida de 180 días es la peor de las cinco (−0.000732 y
−0.001179, esta última con intervalo [−0.00242, −0.00001], que excluye el cero).

La hipótesis de que el fútbol antiguo no informa es razonable y no sobrevive a la
medida: con un modelo dominado por el precio, la cuota ya trae la actualidad: lo que
el pasado añade son partidos con los que estimar la relación entre cuota y resultado.
El panel de abajo, entonces, se lee como el precio que se habría pagado, no como un
canje por decidir. Las cifras están en la sección 2 de `protocol.py`,
junto a los valores por omisión que sostienen.


In [ ]:
print("semividas declaradas (días):", pt.SEMIVIDAS)

# La columna de fecha viaja como texto ISO en la matriz; se convierte aquí, igual
# que hace el protocolo cada vez que la necesita.
ultimo     = pl[-1]
fechas_tr  = pd.to_datetime(feat.loc[ultimo.train_idx, C.DATE])
referencia = pd.to_datetime(feat.loc[ultimo.eval_idx, C.DATE]).min()
edad       = (referencia - fechas_tr).dt.days.clip(lower=0).to_numpy(dtype=float)

# Los pesos no se reimplementan aquí: se piden a la misma función que los aplica
# dentro del pliegue, para que el panel no pueda desviarse de lo que se ejecuta.
def _pesos(h):
    w = pt.recency_weights(fechas_tr, referencia, h)
    return np.ones_like(edad) if w is None else w

filas = []
for h in pt.SEMIVIDAS:
    w = _pesos(h)
    n_ef = w.sum() ** 2 / (w ** 2).sum()
    filas.append({"semivida_dias": "uniforme" if h is None else int(h),
                  "n_entrenamiento": len(w),
                  "n_efectivo": round(n_ef, 1),
                  "fraccion_efectiva": round(n_ef / len(w), 3)})
tabla_kish = pd.DataFrame(filas)
display(tabla_kish)

fig, ax = plt.subplots(figsize=(11, 4))
for h in pt.SEMIVIDAS:
    w = _pesos(h)
    orden = np.argsort(edad)
    ax.plot(edad[orden] / 365.25, w[orden], lw=1.5,
            label="uniforme" if h is None else f"{int(h)} d")
ax.set_xlabel("antigüedad del partido (años)"); ax.set_ylabel("peso relativo")
ax.set_title("Perfiles de decaimiento sobre la ventana del último pliegue")
ax.legend(title="semivida", fontsize=8); plt.tight_layout(); plt.show()

## A.5 · El criterio de lectura, escrito antes de mirar

Se enuncia aquí, antes de la primera tabla, para que no pueda acomodarse a los
resultados. **No es una regla automática**: es el orden en que se leen las columnas y
lo que se considera admisible.

1. **Acierto (`accuracy`).** Primer filtro de cordura. Un conjunto que no llega al
   acierto del favorito de Bet365 difícilmente aporta nada.
2. **RPS.** La métrica principal de calidad probabilística: penaliza el error según lo
   lejos que caiga en el orden local–empate–visitante. Es la que manda cuando dos
   conjuntos empatan en acierto.
3. **Ventaja sobre el listón (`ventaja`).** Rendimiento del modelo menos rendimiento
   del apostante de referencia sobre **los mismos partidos**, con su intervalo por
   remuestreo en bloques. Si el intervalo cruza el cero, la ventaja no es distinguible
   de cero y así se dirá.
4. **Estabilidad temporal.** Una ventaja concentrada en una sola temporada no es una
   ventaja: es una temporada afortunada. Se exige que el signo se sostenga en la
   mayoría de las temporadas del histórico.

Criterios de descarte, también declarados de antemano: conjuntos con **cobertura**
inferior al 10 % de los partidos (apuesta demasiado poco para concluir nada), y
conjuntos cuya ventaja dependa de menos de 150 apuestas.

Cuando dos conjuntos queden empatados dentro del ruido, se prefiere **el más pequeño y
el más interpretable**. La parsimonia es un criterio, no una excusa.

---
# Parte B · Las piezas del banco

Tres cosas hay que conocer antes de empezar a probar: qué familias hay, qué variables
hay agrupadas por dominio, y cuánto cuesta cada ensayo.

## B.1 · Las cuatro familias y su configuración fija

El estudio se reduce a cuatro familias, elegidas por cubrir cuatro maneras distintas de
aprender: lineal y regularizada, ensamblado por promedio, ensamblado por corrección
secuencial y aproximación no lineal continua.

| familia | qué es | por qué está |
|---|---|---|
| `logistica` | regresión logística multinomial con red elástica | referencia interpretable; coeficientes legibles |
| `bosque` | bosque aleatorio | captura no linealidad e interacciones sin ajuste fino |
| `refuerzo_xgb` | potenciación por gradiente | suele dominar en tabular; sensible al sobreajuste |
| `red` | perceptrón multicapa | frontera continua; contrapunto a los árboles |

Durante la búsqueda de variables **la configuración de cada familia se mantiene fija**.
Mover conjunto e hiperparámetros a la vez impide saber a qué se debe una mejora. El
afinado corto llega en la Parte E, sobre el conjunto ya elegido.

In [ ]:
print("familias del estudio:", lb.FAMILIAS, "\n")
for fam in lb.FAMILIAS:
    print(f"{fam:>14} : {lb.HIPER[fam]}")

print("\nespacio del afinado final (Parte E):")
for fam, esp in lb.ESPACIOS_FINAL.items():
    detalle = ", ".join(f"{k}={v}" for k, v in esp.items())
    print(f"{fam:>14} : {detalle}")

## B.2 · El catálogo de dominios

Las variables vienen agrupadas por **dominio**: forma reciente, calidad esperada
(xG), fuerza de plantilla, fatiga y calendario, disponibilidad de efectivos, mercado,
etcétera. Trabajar por dominios es lo que hace manejable la búsqueda manual: en vez de
tantear entre doscientas columnas sueltas, se empieza midiendo bloques con sentido
futbolístico y se mezclan después los que funcionan.

In [ ]:
cat = lb.catalogo(usables)
display(cat)

dom = lb.dominios(usables)
print(f"\ndominios con variables disponibles: {len(dom)}")
for nombre, cols in sorted(dom.items(), key=lambda kv: -len(kv[1])):
    print(f"{nombre:>26} : {len(cols):>3} variables")

In [ ]:
# Ejemplo de lectura del catálogo: las variables de un dominio concreto.
DOMINIO = sorted(dom, key=lambda k: -len(dom[k]))[0]
print(f"dominio '{DOMINIO}' ({len(dom[DOMINIO])} variables):\n")
for c in dom[DOMINIO][:30]:
    print("  ", c)
if len(dom[DOMINIO]) > 30:
    print(f"   ... y {len(dom[DOMINIO]) - 30} más")

## B.3 · Lo que cuesta un ensayo

Un ensayo entrena un modelo por pliegue sobre toda la ventana anterior. El coste crece
con el número de pliegues y con la familia: la logística es de segundos, el bosque y la
red de minutos.

Dos palancas para tantear rápido:

- **`desde=`** recorta el histórico evaluado. `desde="2021/22"` evalúa sólo de esa
  temporada en adelante: sirve para descartar conjuntos malos en una fracción del
  tiempo. La decisión final se toma siempre sobre el histórico completo.
- **La caché de estados.** El preprocesado ya calculado en A.3 se reutiliza en todos
  los ensayos que compartan protocolo. No hace falta hacer nada: es automático. Si se
  cambia el protocolo, `lb.olvida()` limpia la caché.

In [ ]:
import time

t0 = time.perf_counter()
_ = lb.evalua(lb.variables("mercado", columnas=usables)[:8],
              "logistica", nombre="_calibrado_coste",
              evaluar=BLOQUE_BANCO, desde="2021/22",
              registrar=False, verbose=False)
print(f"ensayo de prueba (logística, 8 variables, desde 2021/22): "
      f"{time.perf_counter() - t0:,.1f} s")
print("la caché de estados ya está caliente; los siguientes ensayos con este "
      "protocolo no repiten el preprocesado")

---
# Parte C · El banco de ensayos

**Aquí es donde se trabaja.** Las celdas de esta parte están hechas para editarse y
reejecutarse tantas veces como haga falta. Ninguna de ellas decide nada: todas miden y
enseñan la tabla. La elección la hace quien lee.

## C.1 · Cómo se trabaja

El ciclo es siempre el mismo:

1. **Reconocer el terreno.** `lb.por_dominio(...)` mide cada bloque de variables por
   separado y ordena los dominios por su ventaja sobre el mercado (C.2). Da el mapa de
   dónde hay señal.
2. **Proponer un conjunto.** Se escribe a mano en `CONJUNTO` —normalmente el mejor
   dominio, o la unión de dos o tres— y se mide con `lb.evalua(...)` (C.3).
3. **Ampliar de una en una.** `lb.ronda(base, candidatas)` mide el conjunto base y
   luego el base más cada candidata, y muestra la diferencia que aporta cada una (C.4).
   El analista se queda con la que quiera, o con ninguna.
4. **Comparar lo acumulado.** `lb.compara(...)` pone lado a lado los ensayos que se
   quiera (C.5).
5. **Cambiar de familia.** Cuando el conjunto se estabiliza, `lb.por_familia(...)` lo
   pasa por las cuatro familias (C.6).
6. **Recuperar lo hecho.** `lb.historial()` y `lb.conjunto(nombre)` leen el libro
   acumulativo, que sobrevive al cierre del cuaderno (C.7).

Convenciones útiles:

- **Ponle nombre a todo.** El `nombre` es la clave con la que se recupera un conjunto
  meses después. `"forma_xg_v3"` sirve; `"prueba"` no.
- **Tantea con `desde=`, decide con el bloque de selección entero.** Lo que no se
  hace es tantear sobre el histórico completo: `BLOQUE_BANCO` fija todos los
  ensayos de esta parte en 2019/20-2021/22, porque 2022/23-2024/25 es el bloque
  que C.10 puntúa una sola vez y mirarlo aquí lo gastaría (C.8).
- **Empieza por `logistica`.** Es la más rápida y su orden de mérito entre conjuntos
  suele anticipar el de las demás.
- Los ensayos de tanteo pueden llevar `registrar=False` para no ensuciar el libro.

## C.2 · Reconocimiento: un ensayo por dominio

Mide cada dominio por separado, con la misma familia y el mismo protocolo. La columna
`ventaja` es la diferencia de rendimiento frente al apostante de referencia; `n_variables`
recuerda el tamaño del bloque.

Es un reconocimiento, no un veredicto: un dominio flojo por su cuenta puede ser
excelente como complemento de otro.

In [ ]:
# ---- EDITABLE -------------------------------------------------------------
FAMILIA_RECONOCIMIENTO = "logistica"
DESDE_RECONOCIMIENTO   = "2021/22"      # None = el bloque de selección entero
DOMINIOS_A_MIRAR       = None            # None = todos; o p.ej. ["forma", "xg", "mercado"]
# ---------------------------------------------------------------------------

mapa = lb.por_dominio(FAMILIA_RECONOCIMIENTO,
                      dominios_pedidos=DOMINIOS_A_MIRAR,
                      evaluar=BLOQUE_BANCO, desde=DESDE_RECONOCIMIENTO,
                      columnas=usables,
                      feat=feat)
display(mapa)

no_ev = mapa.attrs.get("no_evaluables", [])
if no_ev:
    print("\ndominios no evaluables:")
    for f in no_ev:
        print(f"  {f['dominio']}: {f['motivo']}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, max(3, 0.42 * len(mapa))))
orden = mapa.sort_values("ventaja")
colores = [PALETA["modelo"] if v > 0 else PALETA["mercado"] for v in orden["ventaja"]]
ax.barh(orden["nombre"], orden["ventaja"], color=colores)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("ventaja sobre el apostante de referencia")
ax.set_title(f"Reconocimiento por dominio · {FAMILIA_RECONOCIMIENTO}"
             + (f" · desde {DESDE_RECONOCIMIENTO}" if DESDE_RECONOCIMIENTO else ""))
plt.tight_layout(); plt.show()

## C.3 — Un conjunto a mano

La celda de trabajo por excelencia. Se escribe el conjunto, se le pone nombre, se
ejecuta. `ensayo.resumen()` da la ficha; `ensayo.tabla()` la abre por temporadas.

Los nombres son los que declara el catálogo (regla 2) y los dominios, los ocho
bloques explicativos de `explain`: `lb.variables` rechaza uno inventado y
`lb.dominios` levanta `KeyError` ante una columna que la matriz no construye, así
que un conjunto escrito a mano se valida al ejecutarlo y no más tarde.

Formas cómodas de construir `CONJUNTO`:

```python
CONJUNTO = lb.variables("forma")                          # un dominio entero
CONJUNTO = lb.variables("forma", "fuerza", "mercado")     # varios dominios
CONJUNTO = lb.conjunto("forma_xg_v2")                     # recuperado del libro
CONJUNTO = lb.conjunto("forma_xg_v2") + ["home_fat_rest_days"]      # y una más
CONJUNTO = ["mkt_p_home", "home_elo_z", "form_npxg_ewm_diff"]       # a mano
```

In [ ]:
# ---- EDITABLE -------------------------------------------------------------
CONJUNTO = lb.variables("mercado", columnas=usables)
FAMILIA  = "logistica"
NOMBRE   = "mercado_solo"
DESDE    = None                          # None = el bloque de selección entero
# ---------------------------------------------------------------------------

ensayo = lb.evalua(CONJUNTO, FAMILIA, nombre=NOMBRE,
                   evaluar=BLOQUE_BANCO, desde=DESDE, feat=feat)
print(ensayo.resumen())

In [ ]:
display(ensayo.tabla())

if ensayo.ausentes:
    print(f"\naviso · {len(ensayo.ausentes)} variables pedidas no existen en la matriz:")
    for c in ensayo.ausentes[:15]:
        print("  ", c)

In [ ]:
t = ensayo.tabla()
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))

x = np.arange(len(t))
ax[0].bar(x, t["accuracy"], width=0.55, color=PALETA["modelo"], label="modelo")
ax[0].axhline(ensayo.accuracy, color=PALETA["apoyo"], ls="--", lw=1.2,
              label="acierto agregado")
ax[0].axhline(ensayo.economia["referencia"].get("acierto", np.nan),
              color=PALETA["mercado"], ls=":", lw=1.4, label="referencia (apuestas)")
ax[0].set_xticks(x); ax[0].set_xticklabels(t["season"], rotation=45, ha="right")
ax[0].set_ylabel("acierto"); ax[0].set_title("Acierto por temporada")
ax[0].legend(fontsize=8)

ax[1].bar(x - 0.2, t["rendimiento_modelo"], width=0.4,
          color=PALETA["modelo"], label="modelo")
ax[1].bar(x + 0.2, t["rendimiento_referencia"], width=0.4,
          color=PALETA["mercado"], label="referencia")
ax[1].axhline(0, color="black", lw=1)
ax[1].set_xticks(x); ax[1].set_xticklabels(t["season"], rotation=45, ha="right")
ax[1].set_ylabel("rendimiento"); ax[1].set_title("Rendimiento por temporada vs. referencia")
ax[1].legend(fontsize=8)

fig.suptitle(f"{ensayo.nombre} · {ensayo.familia} · {len(ensayo.columnas)} variables")
plt.tight_layout(); plt.show()

## C.4 · Ampliar de una en una

`lb.ronda` mide el conjunto base y después el base **más cada candidata por separado**.
Las columnas `d_*` son la diferencia respecto al base: cuánto aporta esa variable.

La tabla llega ordenada por ventaja, pero **no elige**. Una candidata que sube la
ventaja tres milésimas y baja el RPS puede no merecer el coste de una variable más; y
una que mejora sólo en la última temporada probablemente sea ruido. Eso lo juzga quien
lee, no la función.

In [ ]:
# ---- EDITABLE -------------------------------------------------------------
BASE       = CONJUNTO
CANDIDATAS = lb.variables("forma", columnas=usables)[:12]
ETIQUETA   = "ronda1_forma"
# ---------------------------------------------------------------------------

r1 = lb.ronda(BASE, CANDIDATAS, FAMILIA, etiqueta=ETIQUETA,
              evaluar=BLOQUE_BANCO, desde=DESDE, feat=feat)
display(r1)

no_ev = r1.attrs.get("no_evaluables", [])
if no_ev:
    print("\ncandidatas no evaluables:")
    for f in no_ev:
        print(f"  {f['candidata']}: {f['motivo']}")

In [ ]:
sin_base = r1[r1["candidata"] != "(base)"].sort_values("d_ventaja")
fig, ax = plt.subplots(1, 2, figsize=(14, max(3, 0.38 * len(sin_base))), sharey=True)

ax[0].barh(sin_base["candidata"], sin_base["d_ventaja"],
           color=[PALETA["modelo"] if v > 0 else PALETA["mercado"]
                  for v in sin_base["d_ventaja"]])
ax[0].axvline(0, color="black", lw=1)
ax[0].set_xlabel("Δ ventaja"); ax[0].set_title("Aporte económico de cada candidata")

ax[1].barh(sin_base["candidata"], -sin_base["d_rps"],
           color=[PALETA["modelo"] if v > 0 else PALETA["mercado"]
                  for v in -sin_base["d_rps"]])
ax[1].axvline(0, color="black", lw=1)
ax[1].set_xlabel("mejora de RPS (−Δ RPS)")
ax[1].set_title("Aporte probabilístico de cada candidata")

plt.tight_layout(); plt.show()

In [ ]:
# Cuando una candidata convence, se incorpora al conjunto y se vuelve a C.4.
# ---- EDITABLE -------------------------------------------------------------
ELEGIDA = None        # p.ej. "forma_exp_local_5"
NOMBRE_NUEVO = "mercado_forma_v1"
# ---------------------------------------------------------------------------

if ELEGIDA:
    CONJUNTO = list(dict.fromkeys(list(CONJUNTO) + [ELEGIDA]))
    ensayo = lb.evalua(CONJUNTO, FAMILIA, nombre=NOMBRE_NUEVO,
                       evaluar=BLOQUE_BANCO, desde=DESDE, feat=feat)
    print(ensayo.resumen())
else:
    print("no se ha incorporado ninguna candidata en esta ronda")

## C.5 · Comparar ensayos

Pone lado a lado los ensayos que se le pasen. Sirve para decidir entre dos conjuntos
rivales, o para ver si añadir un dominio entero compensa frente a la versión corta.

In [ ]:
# ---- EDITABLE -------------------------------------------------------------
A_COMPARAR = [ensayo]     # p.ej. [ensayo_a, ensayo_b, ensayo_c]
ORDENAR    = "ventaja"    # "ventaja" | "accuracy" | "rps" | "roi" | "n_variables"
# ---------------------------------------------------------------------------

display(lb.compara(*A_COMPARAR, ordenar=ORDENAR))

## C.6 · El mismo conjunto en las cuatro familias

Cuando el conjunto se estabiliza, toca ver si la familia elegida era la adecuada. Cada
familia se entrena con su configuración fija declarada en B.1.

Si una familia no puede evaluarse —típicamente `refuerzo_xgb` sin `xgboost` instalado—
aparece en `attrs["no_aplicables"]` y las demás siguen adelante.

In [ ]:
fam = lb.por_familia(CONJUNTO, evaluar=BLOQUE_BANCO, desde=DESDE, feat=feat)
display(fam)

no_ap = fam.attrs.get("no_aplicables", [])
if no_ap:
    print("\nfamilias no aplicables:")
    for f in no_ap:
        print(f"  {f['familia']}: {f['motivo']}")

In [ ]:
if len(fam) > 1:
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for eje, col, titulo, invertir in [
        (ax[0], "accuracy", "Acierto", False),
        (ax[1], "rps", "RPS (menor es mejor)", True),
        (ax[2], "ventaja", "Ventaja sobre la referencia", False),
    ]:
        d = fam.sort_values(col, ascending=not invertir)
        eje.barh(d["familia"], d[col], color=PALETA["modelo"])
        eje.set_title(titulo)
        if col == "ventaja":
            eje.axvline(0, color="black", lw=1)
    fig.suptitle(f"Las cuatro familias sobre el mismo conjunto ({len(CONJUNTO)} variables)")
    plt.tight_layout(); plt.show()

## C.7 · El libro acumulativo

Cada ensayo con `registrar=True` deja fila en `outputs/models/laboratorio/registro.parquet`:
nombre, familia, variables, protocolo, métricas, economía y un sello criptográfico del
conjunto. El libro **sobrevive a la sesión**: es lo que permite retomar el trabajo días
después sin repetir nada, y es la trazabilidad que la memoria necesita para justificar
por qué se eligió el conjunto que se eligió.

Reejecutar una combinación idéntica **reemplaza** su fila en vez de duplicarla, de modo
que el libro no crece con repeticiones.

In [ ]:
hist = lb.historial()
print(f"ensayos registrados: {len(hist):,}")
display(hist.head(25))

In [ ]:
# Recuperar el conjunto de variables de cualquier ensayo pasado, por su nombre.
# ---- EDITABLE -------------------------------------------------------------
NOMBRE_A_RECUPERAR = NOMBRE
# ---------------------------------------------------------------------------

try:
    recuperado = lb.conjunto(NOMBRE_A_RECUPERAR)
    print(f"'{NOMBRE_A_RECUPERAR}' -> {len(recuperado)} variables")
    for c in recuperado[:25]:
        print("  ", c)
    if len(recuperado) > 25:
        print(f"   ... y {len(recuperado) - 25} más")
except KeyError as e:
    print(e)

In [ ]:
if len(hist) > 1:
    h = hist.sort_values("ventaja")
    fig, ax = plt.subplots(figsize=(11, max(3, 0.34 * len(h))))
    ax.barh(h["nombre"] + "  [" + h["familia"] + "]", h["ventaja"],
            color=[PALETA["modelo"] if v > 0 else PALETA["mercado"] for v in h["ventaja"]])
    ax.axvline(0, color="black", lw=1)
    ax.set_xlabel("ventaja sobre la referencia")
    ax.set_title("Todos los ensayos registrados, de todas las sesiones")
    plt.tight_layout(); plt.show()

---
## C.8 · La búsqueda hacia delante — dónde se elige y dónde se mide

Todo lo anterior compara conjuntos escritos a mano. `seleccion.py` los sustituye por una
búsqueda: prueba `conjunto + una candidata` para cada candidata, adopta la que más
mejora y repite hasta que la mejor mejora del paso ya no llega a la tolerancia.

Una búsqueda así tiene un peligro que no está en el código sino en el método. Si se
prueban doscientas variables sobre los mismos partidos sobre los que después se reporta,
lo que se publica no es el mejor conjunto: es **el máximo de un ruido al que se le han
dado doscientas oportunidades de parecer señal**. El arreglo no consiste en medir mejor,
sino en medir **en otro sitio**. Por eso `protocol.CORTE_SELECCION` parte el bloque de
desarrollo en dos mitades disjuntas que juntas lo cubren entero:

| bloque | temporadas | para qué sirve | ¿citable? |
|---|---|---|---|
| `seleccion` | 2019/20 – 2021/22 | elegir variables, tantas veces como haga falta | no |
| `medicion`  | 2022/23 – 2024/25 | puntuar el conjunto ya cerrado, **una** vez | sí |
| `holdout`   | 2025/26 | sellado hasta el final del trabajo | — |

El bloque **no es un parámetro de la llamada**: `seleccion._guarda_bloque` levanta
`PermissionError` si se intenta cribar sobre medición o medir sobre selección. Un
argumento libre dejaría que una ejecución apresurada arruinase la única cifra citable
del trabajo sin dar ningún error y devolviendo una tabla de aspecto perfectamente
normal.

Quien decide es el **RPS**. Es una regla propia que lee las tres probabilidades de los
~900 partidos del bloque; el acierto sólo mira cuál de las tres es la mayor y tira el
resto, y el rendimiento se calcula sobre el subconjunto apostado, donde un acierto a
6,00 mueve más que veinte apuestas normales. El acierto y la ventaja frente a Bet365 se
anotan en cada paso —están en la traza, columna a columna— pero no votan.

La **criba** es el primer paso: un ensayo por variable, cada una sola. Ordena las
candidatas por lo que dicen por sí mismas, y con `ANCHURA = 0` ese orden es **lectura,
no recorte**: la voraz recorre la lista entera en cada paso. Que una variable caiga a la
cola no demuestra que sea inútil; demuestra que no es útil *sola*, que es una afirmación
más pequeña y es la limitación declarada de lo que la criba mide.


In [ ]:
# La criba: un ensayo por variable, cada una sola, sobre el bloque de SELECCIÓN.
# Coste: una candidata es un protocolo completo. Con la familia logística y la caché
# de estados caliente son minutos; con la red o el refuerzo, horas. Por eso la búsqueda
# se hace con la familia barata y la familia definitiva se decide después, en C.6, ya
# sobre el conjunto elegido.
# ---- EDITABLE -------------------------------------------------------------
FAMILIA_SEL = "logistica"    # la voraz encadena miles de ajustes: familia barata
REUSAR      = True           # True = leer la criba guardada en disco si existe
# ---------------------------------------------------------------------------

pool = sl.candidatas(feat, regime=REGIMEN)
print(f"candidatas de partida : {len(pool):,}   (régimen {REGIMEN})")
print(f"bloque de selección   : temporadas < {pt.CORTE_SELECCION}")
print(f"bloque de medición    : temporadas >= {pt.CORTE_SELECCION}")

if REUSAR and sl.CRIBA.exists():
    tabla_criba = pd.read_parquet(sl.CRIBA)
    print(f"\ncriba leída de disco: {sl.CRIBA.name} ({len(tabla_criba)} variables)")
else:
    tabla_criba = sl.criba(pool, FAMILIA_SEL, feat=feat)

_cols = [c for c in ("orden", "candidata", "accuracy", "rps", "ventaja")
         if c in tabla_criba.columns]
print("\nlas 20 que más dicen por sí solas:")
display(tabla_criba[_cols].head(20))
print("las 10 que menos:")
display(tabla_criba[_cols].tail(10))

## C.9 — La búsqueda

La criba ordena; la búsqueda compone. Y son dos cosas separadas: con `ANCHURA = 0`
—el valor por defecto— la voraz mira **todas** las candidatas en cada paso, de modo
que la criba no recorta el espacio de búsqueda y se lee solo como lo que es, el
retrato de lo que vale cada variable por sí sola. Recortar por mérito individual
tendría sentido si mirarlas todas no cupiese en una tarde, y con un banco de **65**
candidatas sí cabe. De esas 65 la criba mide 62: `ah_line`, `strength_p_home` y
`strength_p_away` **no son evaluables solas**, y no por un defecto del dato sino
porque el preprocesado las retira del estado del pliegue por colinealidad antes de
que el modelo pida columna alguna —`ah_line` correlaciona −0.977 con `mkt_p_home`,
que es lo que cabe esperar de un hándicap, y las dos de fuerza son los miembros
redundantes de un simplejo que suma uno exactamente—, de modo que pedir solo una de
ellas dejaría la matriz vacía. Descontada la semilla, la voraz prueba **59**
candidatas en cada paso.

Mirarlas todas es lo que evita el riesgo propio del recorte por mérito individual:
que una variable que no aporta sola pero sí acompañada no llegue a probarse nunca.

La búsqueda es **flotante**, no solo hacia delante: alterna una pasada de avance con
una de poda hasta que ninguna de las dos cambia nada. El avance puro no puede
deshacer una adopción temprana, y una variable que entró en el paso dos puede quedar
redundante en el paso nueve, cuando ya han entrado otras que dicen lo mismo mejor.
En una logística eso no es inofensivo: dos columnas casi colineales se reparten el
coeficiente, cada una con el doble de error típico, y la lectura del modelo se
ensucia sin que las cifras lo denuncien. La semilla de mercado está protegida de la
poda, porque la pregunta del trabajo es qué se le añade al precio, no si el precio
sobra.

La semilla arranca con el consenso de mercado (`sl.SEMILLA_MERCADO`). La pregunta que
la memoria responde no es «¿qué predice un partido de fútbol?» —a eso ya contesta el
precio— sino «¿qué añade el modelo **por encima** del mercado?», y para responderla
el mercado tiene que estar dentro desde el primer paso, no compitiendo con las demás
por entrar.

El criterio que decide es el **RPS**, por la regla C.1: lee las tres probabilidades
sobre los mil ciento cuarenta partidos del bloque, mientras que el acierto solo mira el
máximo —su ruido es de un punto porcentual, más que casi cualquier diferencia real
entre dos conjuntos— y el ROI se calcula solo sobre los partidos con valor y pagado a
cuota, de modo que un acierto a 6.00 lo mueve más que veinte apuestas normales. El
ROI y el acierto se **informan**, no deciden. Que esa elección importa —y cuánto—
se comprueba aparte, repitiendo la búsqueda guiada por cada criterio y forzando la
adopción en cada paso hasta nueve variables. El resultado no es el que el sentido
común esperaría: **los tres caminos no comparten ni una sola variable**. Puntuados los
tres conjuntos de cinco con las tres métricas, y con la semilla desnuda como
referencia (RPS 0.19317, acierto 52.46 %, ventaja +1.67 %):

| camino | RPS | acierto | ventaja sobre Bet365 |
|---|---|---|---|
| guiado por **RPS** | **0.19254** | 52.81 % | **+12.31 %** |
| guiado por acierto | 0.19337 | **53.51 %** | +0.51 % |
| guiado por ventaja | 0.19321 | 52.46 % | +6.88 % |

Lo que la tabla dice no es que el criterio dé igual, sino que **solo uno de los tres
es bueno en las tres métricas**. El camino del acierto compra 0.70 puntos —ocho
partidos de mil ciento cuarenta— a cambio de un RPS peor que el de la semilla desnuda
y de hundir la ventaja a +0.51 %; el camino de la ventaja consigue **menos** ventaja
que el del RPS, y también empeora la semilla en RPS.

Las dos razones son mecánicas. Perseguir el ROI no da ROI porque el ROI se liquida
sobre el subconjunto en el que la política de valor apuesta, y ese subconjunto se
mueve con cada columna que entra: el primer paso de ese camino ya lo lanza de +1.67 %
a +6.97 % con una sola variable, de modo que lo que la búsqueda optimizaría así no es
el modelo sino la frontera de la apuesta. Y el acierto no tiene resolución para guiar
nada: sobre mil ciento cuarenta partidos solo se mueve a saltos de 0.088 puntos, y
tres de los seis pasos de su camino «mejoran» en exactamente +0.000000, es decir que
adoptan por desempate ciego. Esto es la regla C.1 medida, y no supuesta.

Y hay una segunda condición además del umbral: **la mejora no puede ser de una sola
temporada**. La tolerancia compara medias, y una media sobre tres temporadas la puede
comprar una. En ganancia de RPS por temporada —positiva cuando el RPS baja, que es el
signo con el que la traza la reporta—, la cuarta variable aporta +0.000138 / +0.000817
/ +0.000100 y la quinta +0.000661 / +0.000183 / +0.000010: las dos mejoran en las tres
temporadas. La sexta que la voraz quería adoptar, `n_missing_diff`, aporta −0.000317 /
−0.000174 / +0.001276, es decir que **empeora dos temporadas de tres**, y su media de
+0.000261 —que pasa la tolerancia de sobra— se la debe a 2021/22 sola.

La condición que separa los dos casos no tiene umbral que elegir: se le quita a cada
candidata **su mejor temporada** y se exige que la mejora siga siendo mejora. Quitarla
deja +0.000119 y +0.000096 a las dos que entran, y −0.000246 a la que no. Es una
exigencia de *elegibilidad* y no de parada —una candidata con suerte no es motivo para
dejar de buscar, así que el paso sigue con la siguiente—, y es lo que hace que la
búsqueda responda a lo que el trabajo pregunta: que el conjunto rinda temporada a
temporada, y no solo de media.

¿Y cómo se sabe que el tamaño al que la búsqueda se detiene es el bueno, si la cifra
con la que se detiene es del mismo bloque en el que eligió? Midiéndolo en otro sitio.
Se repite la búsqueda eligiendo solo con 2019/20 y 2020/21 —`corte_seleccion` en
`2021/22`— y forzando la adopción en cada paso (`tolerancia = -1`) hasta las
veinticinco variables, y se puntúa cada prefijo sobre **2021/22, que no participó en
elegirlo**. El RPS toca su mínimo en **cinco** variables (0.19188) y empeora en
diecisiete de los veinte pasos siguientes hasta 0.19912 —los tres retrocesos son de
una cienmilésima—; desde la sexta (0.19300) el conjunto ya rinde peor que la semilla
de mercado sola (0.19199). Esa cola no es ruido, y el mínimo cae donde la búsqueda
para por su cuenta.

Sobre las tres temporadas del bloque, el conjunto de cinco bate al listón de Bet365 en
las tres (+9.01 %, +15.84 % y +11.84 %), mientras que la semilla sola y el conjunto de
cuatro lo baten en una de tres. Son cifras de trabajo, del bloque en el que se eligió,
y no se citan: lo que dicen es que la consistencia que la búsqueda exige variable a
variable se ve también en el conjunto entero.

Y conviene decir con precisión sobre qué descansa la quinta variable, porque no es
sobre el RPS. Su mejora agregada son +0.000285, y el ruido de esa comparación —la
diferencia emparejada partido a partido entre los dos conjuntos, remuestreada por
jornadas completas— tiene un error típico de 0.00026: la mejora son **1.1 errores
típicos**, con intervalo al 95 % de [−0.00023, +0.00078], que incluye el cero. Fuera
de la elección la diferencia es todavía más fina: 0.19189 con cuatro variables frente
a 0.19188 con cinco. El RPS, entonces, **no prefiere cinco variables: es
indiferente**. Lo que separa a los dos conjuntos es la consistencia temporada a
temporada contra el listón —tres de tres frente a una de tres—, y así hay que leerlo,
porque la pregunta del trabajo es exactamente esa.

El remuestreo se desactiva durante la búsqueda (`n_boot=0`): no interviene en
ninguna comparación —se decide con la cifra puntual— y cuesta doscientas
reordenaciones por protocolo. El intervalo se calcula una sola vez, sobre el conjunto
ya cerrado.

La búsqueda se guarda en `outputs/models/seleccion/` **tras cada paso**, no al final:
un cuaderno cerrado a media tarde cuesta un paso, no la tarde entera. Y el paso que
detiene la búsqueda se anota igual que los demás, con `adoptada = False`, porque
forma parte de la evidencia de que la búsqueda paró por algo y no por cansancio.
La búsqueda, con esto, ha hecho su trabajo: **propone** un conjunto de cinco. Quien
decide si ese conjunto se publica es la cuarta puerta, y se pregunta en C.9.1 —fuera del
bloque en el que la búsqueda eligió—, junto con el recorrido del banco entero que
contesta a «¿y si el conjunto bueno fuese mucho más grande?».


In [ ]:
# ---- EDITABLE -------------------------------------------------------------
ANCHURA  = sl.ANCHURA          # 0 = la voraz mira todas las candidatas
MAX_VARS = sl.MAX_VARS         # techo de tamaño del conjunto
VUELTAS  = 6                   # tope de alternancias avance/poda
SEMILLA  = list(sl.SEMILLA_MERCADO)   # el consenso de mercado entra de oficio
# ---------------------------------------------------------------------------

sel = sl.flotante(tabla_criba, FAMILIA_SEL, semilla=SEMILLA, criterio="rps",
                  anchura=ANCHURA, max_vars=MAX_VARS, max_vueltas=VUELTAS,
                  feat=feat, n_boot=0)

traza = sel.traza
_cols = [c for c in ("paso", "fase", "elegida", "n_probadas", "rps", "mejora",
                     "jackknife", "n_inconsistentes", "accuracy", "ventaja",
                     "n_variables", "adoptada")
         if c in traza.columns]
display(traza[_cols])

print(f"vueltas: {sel.vueltas}   podadas: {sel.n_podadas}")
print(f"parada : {sel.motivo_parada}")
print("\nconjunto elegido:")
for i, c in enumerate(sel.conjunto, 1):
    print(f"  {i:>2}. {c}")

In [ ]:
if len(traza) > 1:
    # La curva es la del AVANCE. La poda vuelve a tamaños ya dibujados, con otro
    # conjunto dentro, de modo que superponerla convertiría la línea en un zigzag que
    # no dice nada sobre el tamaño. Y con más de una vuelta el avance se repite igual,
    # así que de cada tamaño vale la primera vez que se probó.
    av = (traza[traza["fase"] == "adelante"]
          .drop_duplicates("n_variables", keep="first")
          .sort_values("n_variables"))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ad = av["adoptada"].astype(bool) if "adoptada" in av else av["paso"] > -1

    ax[0].plot(av["n_variables"], av["rps"], "-o", color=PALETA["modelo"], lw=1.6)
    ax[0].scatter(av.loc[~ad, "n_variables"], av.loc[~ad, "rps"],
                  s=90, facecolors="none", edgecolors=PALETA["mercado"], lw=1.8,
                  zorder=5, label="probada y no adoptada")
    ax[0].set_title("RPS por tamaño del conjunto (menor es mejor)")
    ax[0].set_xlabel("variables en el conjunto"); ax[0].set_ylabel("RPS")
    if (~ad).any():
        ax[0].legend(fontsize=8)

    ax[1].bar(av["n_variables"], av["mejora"],
              color=[PALETA["modelo"] if a else PALETA["apoyo"] for a in ad])
    # La línea de parada es la tolerancia DEL CRITERIO que guió la búsqueda,
    # que es lo que devuelve `sl.tolerancia_de`. Dibujar la del RPS sobre un
    # camino guiado por acierto o por ventaja la pondría pegada al cero: son
    # escalas distintas y 2e-4 en acierto son 0.02 puntos porcentuales.
    _tol = sl.tolerancia_de(sel.criterio)
    ax[1].axhline(_tol, color=PALETA["mercado"], ls="--", lw=1.2,
                  label=f"tolerancia de {sel.criterio}: {_tol:g}")
    ax[1].set_title("Cuánto aportó cada paso")
    ax[1].set_xlabel("variables en el conjunto")
    ax[1].set_ylabel(f"mejora de {sel.criterio}")
    ax[1].legend(fontsize=8)

    # Lo que se ve: el RPS baja mientras entran variables que pagan su sitio y sube
    # en el paso que no lo paga, que es el que detiene la búsqueda y queda dibujado
    # en hueco. Que haya pocos puntos es el resultado, no una curva truncada: con el
    # mercado ya en la semilla, el residuo que el precio no recoge se agota pronto.
    # Y esta curva elige y mide sobre los mismos partidos, así que el mínimo se lee
    # y no decide; el tamaño lo sostiene la lectura de fuera de la elección.
    fig.tight_layout(); plt.show()
else:
    print("la traza tiene un solo paso: no hay curva que dibujar")

---
### C.9.1 · El banco entero, y la cuarta puerta

C.9 para en cinco variables y dibuja una curva de seis puntos. Eso deja dos preguntas
abiertas que la búsqueda no puede contestarse a sí misma, porque las dos se contestan
fuera del sitio en el que eligió: ¿y si el conjunto bueno fuese mucho más grande y la
tolerancia lo estuviese cortando antes de llegar?, y ¿siguen aportando las variables
adoptadas allí donde no se eligieron a sí mismas?

**El banco entero.** Se repite el recorrido forzando la adopción en cada paso
(`tolerancia = -1`, sin la puerta de consistencia y sin techo de tamaño) hasta agotar
las candidatas: cincuenta y ocho pasos, de las tres de la semilla a las sesenta
evaluables. Cada prefijo se puntúa dos veces —**dentro**, sobre los 760 partidos de
2019/20 y 2020/21 con los que se elige, y **fuera**, sobre los 380 de 2021/22, que no
participa en elegir nada—, y las dos lecturas no se parecen:

| tamaño | dentro (760) | fuera (380) |
|---|---|---|
| semilla desnuda, k = 3 | 0.193768 | 0.191987 |
| k = 5 | 0.192868 | **0.191878** ← mínimo fuera |
| k = 9 | **0.191703** ← mínimo dentro | 0.194967 |
| k = 15 | 0.191958 | 0.195947 |
| k = 25 | 0.192666 | 0.198062 |
| k = 60 | 0.200916 | 0.200464 |

La cifra que contesta la pregunta es esta: **el RPS de fuera es peor que el de la
semilla desnuda en 55 de los 57 pasos**, y los dos únicos que lo mejoran son k = 4 y
k = 5 —`out_minshare_max_diff` y `relative_perf_diff`—, exactamente las dos variables
que la búsqueda de C.9 había adoptado por su cuenta. De la sexta a la sesentava no hay
**ni un solo tamaño** que bata al precio desnudo fuera de donde se eligió.

Dentro, en cambio, la curva se porta como uno esperaría de una búsqueda sana: baja de
forma suave hasta un mínimo creíble en nueve variables y solo empieza a deteriorarse
pasadas las quince. Las seis variables que entran entre la sexta y la novena mejoran
todas el ajuste donde se eligen y empeoran todas la medida donde no. Esa tijera es el
sobreajuste de selección, medido: no es que la búsqueda se detuviese pronto por timidez
de la tolerancia, es que no hay ningún tamaño grande que la lectura de fuera sostenga.

El acierto no cambia la lectura, la confirma. Su máximo fuera cae en k = 15 (53.68 %),
con un RPS de 0.195947 que es casi cuatro milésimas peor que el de la semilla: el
acierto premia mover el máximo de las tres probabilidades y es ciego a lo que pase con
las otras dos, que es justo la razón por la que decide el RPS (regla C.1).

**La cuarta puerta.** Las tres primeras —tolerancia, consistencia por temporada y techo
de la nube de ruido— se preguntan todas dentro del bloque de selección, y las dos
variables adoptadas las pasaron las tres: su mejora conjunta era 1.39 veces el máximo
esperable de 59 sorteos simultáneos (z = 2.86), así que no era un artefacto del número
de candidatas probadas. Y aun así, puntuadas donde no se eligieron, **empeoraban** el
modelo: +0.000551 de RPS sobre 1900 partidos, con intervalo [+0.000104, +0.001005], que
excluye el cero. De ahí salió `seleccion.confirma()`, una puerta sin ningún parámetro
que elegir: se mide el conjunto cerrado contra su propia semilla sobre el bloque de
**confirmación** —2017/18 y 2018/19, anteriores a todo lo que la selección miró, de modo
que la regla 6 queda en pie— y se exige solo que el signo sobreviva.

No sobrevive. El conjunto de cinco cuesta **+0.000466** de RPS sobre los 760 partidos de
la confirmación (0.200192 frente a 0.199726 de la semilla sola), con intervalo
[−0.000226, +0.001147] por remuestreo de jornadas completas. La puerta **rechaza**, y
`conjunto.json` queda cerrado sobre las tres variables de precio
(`mkt_p_home`, `mkt_p_draw`, `mkt_p_away`). El fichero guarda las tres cosas —
`conjunto`, `propuesto` y `confirmacion`— precisamente para que un conjunto revertido no
se lea igual que una búsqueda que no encontró nada: la búsqueda encontró dos variables,
y la confirmación las devolvió.

Conviene decir lo que esto significa y lo que no. **No** significa que las 62 variables
restantes no midan nada: significa que lo que miden ya está dentro del precio. El
mercado de apuestas incorpora forma, bajas, Elo, calidad de ocasión y descanso antes de
que el partido empiece, y lo hace con información que el proyecto no tiene —alineaciones
probables, dinero—. Lo que queda después de condicionar por el precio es, en este banco
y con esta familia, indistinguible de cero. Esa es la respuesta medida a la objeción
razonable de que un modelo de tres variables no puede batir a todas las combinaciones
posibles: no las bate por estrechez, las bate porque las otras no añaden señal
**independiente** del precio.

Y lo mismo dice el dinero, que es la otra mitad de la pregunta del trabajo. El recorrido
hasta las veinticinco variables guarda también el rendimiento liquidado sobre 2021/22
contra el listón de Bet365 (−2.75 % esa temporada): la semilla desnuda deja **+18.53 %**
de ROI, el conjunto de cuatro +12.44 % y el de cinco +9.09 %, y desde la **octava** el
modelo ya pierde dinero por debajo del listón, hasta −10.83 % con las veinticinco. La
ventaja sobre el listón cae de **+21.28** puntos a **−8.09** sin un solo repunte
sostenido. No hay, entonces, un tamaño en el que el RPS empeore pero la economía mejore:
las dos curvas de fuera apuntan al mismo sitio, y es el sitio donde la búsqueda paró.

> Ninguna cifra de esta sección es citable (regla 5): todas salen del bloque de
> selección o del de confirmación, que son los bloques donde se decide. La cifra que va
> a la memoria es la de C.10, sobre el bloque de medición, con el conjunto ya cerrado.


In [ ]:
import json   # el cuaderno no lo importa en la cabecera y aqui se leen dos JSON

# Regla 8: las dos lecturas de FUERA se leen de disco, no se recalculan aquí. Las
# produjeron dos pasadas de `seleccion.flotante` con la adopción forzada, y viven bajo
# `outputs/`, que no se versiona: si no están, la celda lo dice y el texto de arriba
# conserva las cifras.
g = sl.carga() if sl.CONJUNTO.exists() else None
if g is None:
    print(f"sin conjunto guardado en {sl.CONJUNTO}")
else:
    print("cerrado en disco :", ", ".join(g["conjunto"]))
    print("propuesto por C.9:", ", ".join(g.get("propuesto") or g["conjunto"]))
    conf = g.get("confirmacion") or {}
    if conf.get("n"):
        veredicto = "CONFIRMA" if g.get("confirmado") else "RECHAZA"
        print(f"cuarta puerta    : {veredicto} - cuesta {conf['cuesta']:+.6f} de RPS "
              f"sobre {conf['n']} partidos de {', '.join(conf['temporadas'])}, "
              f"intervalo [{conf['inf']:+.6f}, {conf['sup']:+.6f}]")
    else:
        print("cuarta puerta    :", conf.get("motivo", "no preguntada"))

if sl.CAMINO_65.exists():
    cam = pd.read_csv(sl.CAMINO_65)
    base = float(cam.sort_values("k")["rps_fuera"].iloc[0])
    peores = int((cam["rps_fuera"] > base).sum())
    k_in = int(cam.loc[cam["rps_dentro"].idxmin(), "k"])
    k_out = int(cam.loc[cam["rps_fuera"].idxmin(), "k"])
    n_in, n_out = int(cam["n_dentro"].iloc[0]), int(cam["n_fuera"].iloc[0])

    fig, ax = plt.subplots(figsize=(12, 4.2))
    ax.plot(cam["k"], cam["rps_dentro"], "-o", ms=3.5, lw=1.4,
            color=PALETA["modelo"], label=f"dentro: elige y mide ({n_in} partidos)")
    ax.plot(cam["k"], cam["rps_fuera"], "-s", ms=3.5, lw=1.4,
            color=PALETA["mercado"], label=f"fuera: 2021/22 ({n_out} partidos)")
    ax.axhline(base, color=PALETA["apoyo"], ls="--", lw=1.2,
               label=f"semilla desnuda, fuera ({base:.5f})")
    ax.axvline(k_in, color=PALETA["modelo"], ls=":", lw=1.0)
    ax.axvline(k_out, color=PALETA["mercado"], ls=":", lw=1.0)
    ax.set_title("El banco entero con adopción forzada: dentro baja, fuera no")
    ax.set_xlabel("variables en el conjunto")
    ax.set_ylabel("RPS (menor es mejor)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    mejoran = ", ".join(f"k={int(k)}" for k in cam.loc[cam["rps_fuera"] < base, "k"])
    print(f"minimo dentro: k = {k_in} ({cam['rps_dentro'].min():.6f})    "
          f"minimo fuera: k = {k_out} ({cam['rps_fuera'].min():.6f})")
    print(f"fuera es PEOR que la semilla desnuda en {peores} de {len(cam) - 1} pasos; "
          f"la mejoran solo: {mejoran or 'ninguno'}")
else:
    print(f"\nsin recorrido del banco entero en {sl.CAMINO_65}")

if sl.CAMINO_FUERA.exists():
    # El mismo recorrido hasta 25 variables, que es el único que guarda la liquidación
    # económica de fuera. Se lee de disco por la regla 8: la produjo `backtest` dentro
    # de la pasada de selección, no esta celda.
    fuera = pd.DataFrame(json.loads(sl.CAMINO_FUERA.read_text(encoding="utf-8"))["filas"])
    ref = float(fuera["roi_ref_fuera"].iloc[0])

    fig, ax = plt.subplots(figsize=(12, 3.8))
    ax.plot(fuera["k"], 100 * fuera["roi_fuera"], "-o", ms=3.5, lw=1.4,
            color=PALETA["modelo"], label="ROI del modelo sobre 2021/22")
    ax.axhline(100 * ref, color=PALETA["mercado"], ls="--", lw=1.2,
               label=f"listón de Bet365 ({100 * ref:+.2f} %)")
    ax.axhline(0, color=PALETA["apoyo"], lw=0.8)
    ax.set_title("La economía de fuera, por tamaño del conjunto")
    ax.set_xlabel("variables en el conjunto")
    ax.set_ylabel("ROI (%)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    cruce = fuera.loc[fuera["ventaja_fuera"] < 0, "k"]
    print(f"ROI fuera: {100 * fuera['roi_fuera'].iloc[0]:+.2f} % con la semilla "
          f"desnuda -> {100 * fuera['roi_fuera'].iloc[-1]:+.2f} % con "
          f"{int(fuera['k'].iloc[-1])} variables")
    print("la ventaja sobre el listón se vuelve negativa en k = "
          f"{int(cruce.iloc[0]) if len(cruce) else 'nunca'} y no se recupera")
else:
    print(f"sin recorrido economico de fuera en {sl.CAMINO_FUERA}")


### C.9.2 · ¿Y miden algo, las otras? El modelo sin precio

Todo lo de C.9 y C.9.1 mide las variables **como añadido** al precio: la búsqueda parte
de la semilla de mercado y pregunta qué suma cada candidata. El resultado fue nulo, y un
resultado nulo así admite dos lecturas que no se parecen en nada:

1. las variables de juego de este banco no miden casi nada; o
2. miden mucho, y el precio ya lo ha incorporado antes de que el partido empiece.

La diferencia importa, y no es retórica: la primera lectura dice que el trabajo de
extracción y de variables fue en balde, y la segunda dice que midió bien algo que el
mercado también mide, lo cual es un hallazgo sobre el mercado y no un fracaso del banco.
Las dos lecturas son indistinguibles mientras el precio esté dentro del modelo, porque
condicionar por el precio es exactamente lo que las confunde.

Se separan quitándoselo. Se ajustan las **59 variables de juego sin ninguna columna de
precio**, sobre el bloque de selección, con dos referencias que acotan el problema por
arriba y por abajo: la **previa de frecuencias** —la distribución de clases de la propia
ventana, que no mira ni quién juega— como suelo, y la **semilla de mercado** como techo.
La columna `camino` sitúa cada conjunto en ese recorrido: 0 es la previa, 1 es la
semilla.

| conjunto | k | RPS | acierto | ROI | `camino` |
|---|---|---|---|---|---|
| semilla de mercado | 3 | **0.193174** | 52.46 % | −0.50 % | 1.000 |
| dominio de mercado entero | 6 | 0.193623 | 52.28 % | −3.15 % | 0.985 |
| **solo `fuerza`** | 9 | **0.198001** | 51.75 % | −3.80 % | **0.841** |
| las 65 juntas | 65 | 0.200669 | 50.18 % | −2.34 % | 0.753 |
| **las 59 de juego, sin precio** | 59 | **0.202853** | 49.65 % | −4.16 % | **0.681** |
| solo `forma` | 26 | 0.202919 | 49.39 % | −12.38 % | 0.679 |
| solo `clasificacion` | 7 | 0.207586 | 48.16 % | −9.34 % | 0.525 |
| solo `historial` | 3 | 0.213901 | 46.49 % | −13.80 % | 0.317 |
| solo `fatiga` | 6 | 0.219168 | 44.74 % | −10.40 % | 0.143 |
| solo `bajas` | 4 | 0.223182 | 43.51 % | −10.10 % | 0.011 |
| previa de frecuencias | — | 0.223518 | 43.60 % | — | 0.000 |
| solo `contexto` | 4 | 0.224266 | 43.60 % | −7.18 % | −0.025 |

**La segunda lectura es la correcta.** Las 59 variables de juego, sin ver un solo precio,
recorren el **68.1 %** del camino que separa la previa de frecuencias del mercado
(0.020664 de los 0.030343 de hueco). Y el acierto, que es una métrica independiente y
mucho más basta, dice **68.3 %** —6.05 de los 8.86 puntos porcentuales—. Que dos
métricas que no comparten construcción coincidan en la primera décima es, en este
cuaderno, la coincidencia más tranquilizadora que se ha encontrado: el banco de variables
contiene de verdad dos tercios de la información con la que el mercado fija su precio, y
la reconstruye desde cero, sin verlo.

Lo que no contiene es el tercio que falta, y ahí la respuesta honesta es que el proyecto
no puede tenerlo: son alineaciones confirmadas, noticias de vestuario, y el dinero de los
propios apostantes, que es el mecanismo por el que el precio agrega información privada
que ninguna fuente pública publica. El techo de las variables públicas está, en este
banco, dos tercios del camino.

**Y hay un segundo hallazgo, que cambia la interpretación de C.9.1.** El dominio de
`fuerza` solo —nueve columnas, Elo y valor de plantilla— llega al **0.841 del camino con
RPS 0.198001, y bate a las 59 juntas** (0.202853). Añadir las otras cincuenta no mejora:
empeora. Ese 68.1 % no es, por tanto, el techo del banco de juego, sino el techo de un
**ajuste de 59 variables** sobre 1140 partidos: lo que sobra es varianza. La lectura
mecánica de por qué la voraz no encontró nada queda entonces completa, y no es que no
hubiera señal —la hay, y mucha— sino que **cada variable que entra cuesta más varianza de
la que trae de señal nueva**, porque la señal que trae ya estaba en el precio y la
varianza es toda suya.

La fila de `todo` lo dice del modo más crudo: las 65 juntas dan **0.200669**, que es
**0.0075 peor que las tres del precio solas**. Cincuenta y nueve columnas de juego,
añadidas a un precio que ya acierta, no solo no aportan: ahogan lo que el precio sabía.
Con eso, la escalera de C.9 y la parada de la búsqueda dejan de ser un resultado
decepcionante y pasan a ser el resultado correcto.

La escalera por dominios, por último, se ordena como ordenaría cualquiera que entienda de
fútbol, y eso es una comprobación de cordura del pipeline entero: `fuerza` (0.841) por
delante de `forma` (0.679), después la `clasificacion` (0.525), el `historial` directo
(0.317), la `fatiga` (0.143), y al final `bajas` (0.011) y `contexto` (−0.025), que por sí
solos **no se distinguen de no saber nada**. Que `bajas` esté pegado al suelo encaja con
lo que la cuarta puerta ya había rechazado: `out_minshare_max_diff`, una de las dos
variables que la voraz adoptó y la confirmación devolvió, es justamente de ese dominio.

El dinero no rescata a ninguno: todos los modelos sin precio pierden, de −4.16 % a
−13.80 %, contra el −2.18 % del listón de Bet365 en este bloque. Solo los dos que llevan
precio dentro se acercan a no perder. Aquí no hay nada que arbitrar entre el RPS y el
ROI, porque las dos métricas apuntan al mismo sitio. Cuando sí discrepan —y discrepan en
C.9.4— la regla 4 del proyecto decía que mandaba el RPS, y **Manuel la levantó el
19-oct-2026 para la elección del modelo publicado**: ahí manda el ROI. El alcance es esa
elección y nada más; todo lo que se lee en esta sección se sigue ordenando por RPS.

> Ninguna cifra de esta sección es citable (regla 5): sale del bloque de selección, que
> es donde se decide. Lo que esta sección aporta a la memoria no es una cifra sino la
> **interpretación** del resultado negativo de C.9, y esa interpretación ya no es una
> conjetura razonable sino una medida. La produce `_sinprecio.py` y la deja en
> `seleccion.SINPRECIO`.


In [ ]:
# Regla 8: la tabla se lee de disco, no se recalcula. La produjo `_sinprecio.py` con
# doce protocolos completos sobre el bloque de selección, y vive bajo `outputs/`, que no
# se versiona: si no está, la celda lo dice y el texto de arriba conserva las cifras.
if not sl.SINPRECIO.exists():
    print(f"sin descomposicion en {sl.SINPRECIO}")
else:
    sp = pd.read_csv(sl.SINPRECIO).sort_values("camino", ascending=False)

    suelo = sp.loc[sp["conjunto"] == "previa_frecuencias"].iloc[0]
    techo = sp.loc[sp["conjunto"] == "semilla_mercado"].iloc[0]
    juego = sp.loc[sp["conjunto"] == "juego_sin_precio"].iloc[0]
    hueco = float(suelo["rps"]) - float(techo["rps"])
    d_acc = float(techo["accuracy"]) - float(suelo["accuracy"])

    print(f"hueco previa -> semilla      : {hueco:.6f} de RPS "
          f"({100 * d_acc:.2f} pp de acierto)")
    print(f"lo recorren las 59 de juego  : {100 * float(juego['camino']):.1f} % en RPS, "
          f"{100 * (float(juego['accuracy']) - float(suelo['accuracy'])) / d_acc:.1f} %"
          f" en acierto")
    mejor = sp.loc[sp["conjunto"].str.startswith("solo_"), :].iloc[0]
    print(f"mejor dominio por si solo    : {mejor['conjunto'][5:]} "
          f"(k={int(mejor['k'])}, RPS {mejor['rps']:.6f}, "
          f"camino {100 * float(mejor['camino']):.1f} %)"
          f"{'  <- bate a las 59 juntas' if mejor['rps'] < juego['rps'] else ''}")
    todo = sp.loc[sp["conjunto"] == "todo"].iloc[0]
    print(f"las 65 juntas vs la semilla  : {float(todo['rps']) - float(techo['rps']):+.6f}"
          f" de RPS (el precio se ahoga)")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2))

    vis = sp[sp["conjunto"] != "previa_frecuencias"]
    col = [PALETA["mercado"] if c in ("semilla_mercado", "mercado_entero")
           else PALETA["modelo"] if c in ("juego_sin_precio", "todo")
           else PALETA["apoyo"] for c in vis["conjunto"]]
    ax1.barh(vis["conjunto"], vis["camino"], color=col)
    ax1.axvline(0, color="black", lw=1)
    ax1.axvline(1, color=PALETA["mercado"], lw=1, ls="--")
    ax1.set_xlabel("camino recorrido entre la previa (0) y la semilla de mercado (1)")
    ax1.set_title("Cuanto del precio reconstruye cada bloque, sin verlo")
    ax1.invert_yaxis()

    # La misma lectura con el acierto, que no comparte construccion con el RPS. Si las
    # dos nubes caen sobre la diagonal, el orden no es un artefacto de la metrica.
    acc_cam = (vis["accuracy"] - float(suelo["accuracy"])) / d_acc
    ax2.scatter(vis["camino"], acc_cam, color=PALETA["modelo"], s=55, zorder=3)
    lim = [min(-0.1, float(vis["camino"].min())), 1.08]
    ax2.plot(lim, lim, color=PALETA["apoyo"], lw=1, ls="--", zorder=1)
    for _, r in vis.iterrows():
        ax2.annotate(str(r["conjunto"]).replace("solo_", ""),
                     (r["camino"], (r["accuracy"] - float(suelo["accuracy"])) / d_acc),
                     fontsize=7.5, xytext=(4, -3), textcoords="offset points")
    ax2.set_xlabel("camino segun RPS")
    ax2.set_ylabel("camino segun acierto")
    ax2.set_title("Dos metricas independientes, el mismo orden")
    plt.tight_layout(); plt.show()


### C.9.3 · ¿Y el mercado MÁS lo mejor del resto?

Es la objeción directa, y la que cualquiera haría en la defensa. C.9 contestó «nada
mejora al precio», pero ese conjunto —el mercado junto a las mejores del resto— **la
voraz no podía encontrarlo aunque existiera**, y por dos limitaciones que son del
procedimiento y no de los datos.

**Una: la penalización estaba clavada.** `laboratorio.HIPER["logistica"]` fija
`C=0.3, l1_ratio=0.0` para toda la búsqueda. El propio módulo declara el precio de esa
decisión —«un conjunto puede parecer peor que otro por llevar una configuración que
no le convenía»— y lo remite a `afina()`, pero `afina()` solo afina el conjunto que
**ya ganó**, así que la compensación nunca llegó a los conjuntos que la necesitaban.
Y la necesitaban:
las **mismas 65 columnas** pasan de 0.200669 con `C=0.3, L2` a **0.194557** con
`C=0.03, L1`. Son **0.0061 de RPS recuperados por la penalización sola, sin tocar una
variable**. Dicho de otro modo: la voraz comparaba k=3 contra k=12 con un castigo de
varianza calibrado para k=3, y buena parte de lo que medía era ese castigo y no la falta
de señal.

**Dos: la voraz es miope.** Añade de una en una y se queda con la que más mejora
sola. Si un BLOQUE ayuda y ninguno de sus miembros ayuda por separado, no lo encuentra
nunca. Y eso no es hipotético aquí: C.9.2 midió que `fuerza` —nueve columnas— **bate a
las 59 de juego juntas**.

Se corrigen las dos a la vez. Cada conjunto se mide con **su** mejor penalización de
cinco —incluida la semilla, para que la pelea sea limpia en los dos ejes— y los
conjuntos son dominios **enteros**, más la escalera acumulativa, que es la pregunta en
su forma literal: el mercado más los dominios de mejor a peor. El orden de la escalera
no se elige mirando estas cifras —eso sería la voraz otra vez— sino el que C.9.2 ya
midió **sin el precio delante**, donde el precio no puede haber tapado a nadie.

| conjunto | k | RPS (su mejor penalización) | gana a la semilla | intervalo |
|---|---|---|---|---|
| **semilla+`bajas`** | 7 | **0.192958** | **+0.000211** | [−0.000377, +0.000814] |
| semilla (listón) | 3 | 0.193169 | — | — |
| semilla+`historial` | 6 | 0.193225 | −0.000057 | [−0.000327, +0.000221] |
| semilla+`contexto` | 7 | 0.193647 | −0.000478 | [−0.001349, +0.000436] |
| semilla+`fatiga` | 9 | 0.193683 | −0.000514 | [−0.001471, +0.000457] |
| semilla+`clasificacion` | 10 | 0.193694 | −0.000525 | [−0.001146, +0.000112] |
| semilla+`fuerza` | 12 | 0.193949 | −0.000780 | [−0.001584, +0.000070] |
| semilla+`forma` | 29 | 0.194065 | −0.000897 | [−0.001737, −0.000099] |

**La corrección estrecha la distancia de verdad, y conviene decirlo en voz alta:**
`semilla+fuerza` pasa de perder 0.001081 a perder **0.000780**. La comparación original
era injusta y ahora no lo es. Pero el veredicto no se mueve, y la escalera acumulativa
lo dice del modo más crudo —**monótonamente peor**, 0.193949 con `fuerza`, 0.194226 con
`forma`, hasta **0.194399** con las 62, con intervalos que excluyen el cero desde
`forma` en adelante—. No es ruido, es daño.

Queda una ruta más, la cuarta, y es la más generosa con la hipótesis: ordenar las 59 de
juego **por mérito propio** (k=1, sin el precio delante, donde ninguna puede quedar
tapada por el mercado) y juntar la semilla con las N mejores. El orden sale como debe
—`elo_z_diff` 0.198131, `strength_attack_diff` 0.203888, `cum_xgd_pg_diff` 0.204455— y
**ningún corte gana**: top3 −0.000259, top5 −0.000252, top8 −0.000335, top12 −0.000593,
top20 −0.000813, top30 −0.000928. Ordenar 59 variables sobre los mismos 1140 partidos y
quedarse con la cabeza es sesgo de máximo en estado puro, y se hace a propósito: si ni
con esa ventaja gana a la semilla, la respuesta es firme.

**Los dos conjuntos que sí ganaron pasan por la cuarta puerta, y los dos caen con el
signo invertido**, que es la firma de haber escogido el máximo de treinta y cinco
configuraciones. `semilla+bajas` gana 0.000211 donde se eligió y **pierde 0.000312 donde
no** —y es, no por casualidad, el mismo dominio de `out_minshare_max_diff`, una de las
dos variables que la voraz adoptó y esta misma puerta devolvió—. `semilla+historial`
empata en RPS y gana **+0.61 pp de acierto**, que parecen algo hasta que se miden contra
su propio error típico: el del acierto sobre 1140 partidos es **1.48 pp**, así que esa
ganancia son **0.41 errores típicos**, es decir nada; y fuera pierde 0.000586 con
intervalo [+0.000075, +0.001115], que ya excluye el cero.

Para que la puerta pudiera juzgarlos hubo que darle un argumento que no tenía:
`seleccion.confirma()` acepta ahora `hiper`. Un conjunto adoptado bajo `C=0.1, L1` y
confirmado bajo `C=0.3, L2` se estaría puntuando con un castigo de varianza que su
elección no tuvo, y el rechazo no distinguiría la falta de señal del cambio de
configuración. La semilla y el conjunto se puntúan siempre con la misma, que es lo que
mantiene la diferencia emparejada interpretable.

> Ninguna cifra de esta sección es citable (regla 5): sale del bloque de selección, y
> además elige la penalización mirando los mismos partidos —cinco miradas por conjunto—.
> Ese sesgo es simétrico entre conjuntos y por tanto no favorece a los grandes, que es
> lo que se pone a prueba, pero sesgo es. La puerta de verdad es
> `seleccion.confirma()` sobre 2017/18–2018/19. Las tablas las producen `_bloques.py`
> y `_mejores.py`, y viven en `seleccion.BLOQUES`, `MEJORES` y `MEJORES_ORDEN`.


In [ ]:
# Regla 8: las dos tablas se leen de disco, no se recalculan. Las produjeron
# `_bloques.py` (quince conjuntos x cinco penalizaciones) y `_mejores.py` (las 59 solas
# mas los cortes acumulados), y viven bajo `outputs/`, que no se versiona.
if not (sl.BLOQUES.exists() and sl.MEJORES.exists()):
    print(f"sin el cuadro de bloques en {sl.BLOQUES}")
else:
    bq = pd.read_csv(sl.BLOQUES)
    mj = pd.read_csv(sl.MEJORES)

    def _mejor(t):
        """Cada conjunto en SU mejor penalizacion, que es lo que la busqueda no hizo."""
        return (t.loc[t.groupby("conjunto")["rps"].idxmin()]
                 .sort_values("rps").reset_index(drop=True))

    bqm, mjm = _mejor(bq), _mejor(mj)
    sem = float(bqm.loc[bqm["conjunto"] == "semilla", "rps"].iloc[0])

    # El precio de haber clavado la penalizacion, medido en este mismo cuadro: cuanto
    # recupera cada conjunto al dejarle elegir la suya en vez de la de la busqueda.
    fijo = (bq.loc[(bq["C"] == 0.3) & (bq["l1_ratio"] == 0.0)]
              .set_index("conjunto")["rps"])
    rec = (fijo - bqm.set_index("conjunto")["rps"]).sort_values(ascending=False)
    print(f"semilla en su mejor penalizacion : {sem:.6f}")
    print(f"lo que recupera la penalizacion  : hasta {rec.max():.6f} "
          f"({rec.idxmax()}), y solo {rec.get('semilla', float('nan')):.6f} en la "
          f"semilla")
    print("   -> la rejilla ayuda a los conjuntos GRANDES y no al pequeno: por eso la "
          "comparacion original era injusta")

    gan = bqm.loc[bqm["gana"] > 0, "conjunto"].tolist()
    print(f"conjuntos que ganan a la semilla : {', '.join(gan) if gan else 'ninguno'}"
          f"  (y la cuarta puerta los rechaza, con el signo al reves)")
    peor = mjm.loc[mjm["conjunto"] != "semilla", "gana"].max()
    print(f"el mejor de los cortes top-N     : {peor:+.6f} de RPS, tampoco gana")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2))

    # Izquierda: cada dominio entero contra la semilla, con su intervalo por jornadas.
    bl = bqm[bqm["conjunto"].str.startswith("semilla+")].sort_values("gana")
    col = [PALETA["modelo"] if g > 0 else PALETA["apoyo"] for g in bl["gana"]]
    ax1.barh(bl["conjunto"].str.replace("semilla+", ""), bl["gana"], color=col)
    ax1.errorbar(bl["gana"], range(len(bl)),
                 xerr=[bl["gana"] - bl["inf"], bl["sup"] - bl["gana"]],
                 fmt="none", ecolor="black", elinewidth=1, capsize=3)
    ax1.axvline(0, color=PALETA["mercado"], lw=1.2)
    ax1.set_xlabel("RPS que le quita a la semilla (positivo = gana)")
    ax1.set_title("El mercado mas cada dominio entero, afinado")

    # Derecha: las dos escaleras. La acumulativa por dominios y los cortes top-N, ambas
    # contra el liston de la semilla. Ninguna baja de la linea.
    ORD = ["fuerza", "forma", "clasificacion", "historial", "fatiga", "bajas",
           "contexto"]
    paso = {f"escalera+{d}": i for i, d in enumerate(ORD)}
    # `esc` y no `es`: el cuaderno importa `estimators as es` en la cabecera, y
    # `test_referencias` resuelve los atributos de los alias de modulo contra el .py
    # real. Una variable local llamada `es` convierte `es.iterrows()` en una
    # referencia rota a `estimators.iterrows`, y la bateria lo denuncia con razon.
    esc = (bqm[bqm["conjunto"].isin(paso)]
           .assign(paso=lambda t: t["conjunto"].map(paso)).sort_values("paso"))
    tn = mjm[mjm["conjunto"].str.startswith("semilla+top")].sort_values("k")
    ax2.axhline(sem, color=PALETA["mercado"], lw=1.2, ls="--",
                label=f"semilla sola ({sem:.6f})")
    ax2.plot(esc["k"], esc["rps"], "o-", color=PALETA["modelo"],
             label="escalera por dominios enteros")
    ax2.plot(tn["k"], tn["rps"], "s-", color=PALETA["apoyo"],
             label="semilla + las N mejores por merito propio")
    for _, r in esc.iterrows():
        ax2.annotate(str(r["conjunto"]).replace("escalera+", ""), (r["k"], r["rps"]),
                     fontsize=7.5, xytext=(3, 4), textcoords="offset points")
    ax2.set_xlabel("k (numero de columnas del conjunto)")
    ax2.set_ylabel("RPS en su mejor penalizacion")
    ax2.set_title("Anadir no baja la linea, por ninguna de las dos rutas")
    ax2.legend(fontsize=8)
    plt.tight_layout(); plt.show()


### C.9.4 · La novena ruta: mezclar en vez de añadir, y la que se publicó

Las ocho rutas de C.9 a C.9.3 preguntan todas lo mismo de la misma manera: **qué
columnas meter en un ajuste**. Y las ocho dan el mismo resultado nulo con la misma
firma —el signo se invierte fuera del bloque que eligió—, que es lo que ocurre cuando lo
que se ha medido es el máximo de una colección y no una ventaja. C.9.3 dejó el
diagnóstico escrito: cada variable que entra cuesta más varianza de la que trae de señal
nueva, porque su señal ya estaba en el precio y la varianza es toda suya.

Si el problema es la varianza del ajuste conjunto, la salida no es elegir mejor las
columnas: es **no ajustarlas juntas**. Se ajustan dos modelos **separados** —uno sobre
las tres columnas de precio, otro sobre las 59 de juego— y se combinan sus
probabilidades como una **mezcla logit**, también llamada *log-opinion pool*:

$$p \propto p_{\text{mercado}}^{\,w} \cdot p_{\text{juego}}^{\,1-w}$$

renormalizando para que la terna sume uno. El modelo de juego no compite por entrar en
la matriz del precio: opina aparte, y `w` gradúa cuánto se le escucha. Lo que cambia
respecto a las ocho rutas anteriores es el **tamaño de lo que se elige**: aquí hay **un
solo parámetro**, frente a los 2^65 conjuntos entre los que elegía la voraz. Apenas hay
nada que sobreajustar, y eso es lo que se espera que se note fuera.

**Y se nota.** La diferencia emparejada contra la semilla, para `w=0.85` y por bloque:

| bloque | n | acierto | ROI | RPS |
|---|---|---|---|---|
| `seleccion` (eligió) | 1140 | +0.44 pp [−0.58, +1.46] | +1.80 pp [−1.02, +4.63] | −0.000181 |
| `medicion` (no eligió) | 1140 | **+0.35 pp** [−0.42, +1.12] | **+1.12 pp** [−1.00, +3.25] | −0.000220 |
| `confirmacion` (no eligió) | 760 | **+0.39 pp** [−0.67, +1.46] | **+1.51 pp** [−1.43, +4.44] | −0.000989 |

Es la **única de las nueve rutas cuyo signo no se invierte**: el acierto y el ROI
repiten en los dos bloques que no participaron en elegir el peso. Y dice dos cosas más
que hay que leer juntas, porque separadas engañan:

- **Ninguna diferencia de fuera excluye el cero.** Que el signo repita no es que la
  ventaja esté demostrada; es que **no se desmorona**, que es lo que hacían las otras
  ocho. La distinción importa y no se puede ascender a significación.
- **El RPS es negativo en los tres bloques.** La mezcla **empeora el RPS**, siempre, con
  todos los pesos. No es un efecto de muestra: es lo que hace la mezcla.

De ahí que adoptarla sea una decisión y no una lectura. **Levanta la regla 4 del
proyecto** —cuando el ROI y el RPS discrepan, manda el RPS—, y la levantó **Manuel el
19-oct-2026**: «quiero que el ROI mande, no el RPS». El alcance es la elección del
modelo publicado y nada más.

**El peso se elige por ACIERTO, y es el hallazgo de método de toda la Parte C.** La
curva da tres óptimos distintos según con qué se mire sobre el bloque de selección:
`w_rps = 1.00` —es decir, no mezclar—, `w_acierto = 0.85` y `w_roi = 0.35`. Elegir por
ROI da w=0.35 y elegir por acierto da w=0.85, y **fuera gana el segundo en las dos
convenciones de liquidación**, que hay que nombrar porque sus niveles no son
intercambiables: a la mejor cuota entre casas (columna `roi_max`, promediada por
partidos sobre `medicion` más `confirmacion`) **+2.51 % contra +1.38 %**; de ventaja
sobre el listón a cierre promediado (columna `vent`, la emparejada) **+1.08 % contra
−0.05 %**. El motivo es de tamaño de error: el ROI sobre 1140 partidos tiene un error
típico de unos **3 pp**, de modo que la diferencia entre dos pesos es ruido frente a su
propia incertidumbre. **Seleccionar por ROI es seleccionar ruido**, aunque el ROI sea la
cifra que se quiere maximizar. La regla que queda es: **se selecciona por acierto y se
reporta ROI.**

Un aviso sobre la cifra de ROI de esta sección, para que no se confunda con la
publicada: aquí el ROI es el del banco, política **`valor`** con umbral, y la diferencia
va **contra la semilla**. La que publica `build_models.py` es política **`favorito`** —una
unidad plana al signo del modelo en todos los partidos con precio— y la diferencia va
**contra el listón de Bet365**. No son el mismo número y no deben citarse
intercambiados.

La misma mezcla con `fuerza` en lugar de las 59 de juego —el dominio que C.9.2 midió
como el mejor del banco— **no mejora**: su `w_acierto` es 0.65 y el signo del acierto no
repite fuera. El rival de la mezcla es el banco de juego entero, no su mejor dominio.

> Las cifras de `seleccion` de esta sección no son citables (regla 5). Las de `medicion`
> y `confirmacion` sí lo son, y son justamente las que sostienen la decisión: están
> fuera de la elección del peso. Lo produce `_mezcla.py` y lo deja en `mezcla.csv`,
> `mezcla_par.csv` y `mezcla_resumen.json`, bajo `outputs/`, que no se versiona.

In [ ]:
# Regla 8: las dos tablas se leen de disco, no se recalculan aqui. Las produjo
# `_mezcla.py` (el barrido de w por bloque, mas la diferencia EMPAREJADA contra la
# semilla) y viven bajo `outputs/`, que no se versiona.
if not (sl.MEZCLA.exists() and sl.MEZCLA_PAR.exists()):
    print(f"sin la curva de la mezcla en {sl.MEZCLA}")
else:
    cur = pd.read_csv(sl.MEZCLA)
    par = pd.read_csv(sl.MEZCLA_PAR)
    W_PUB = 0.85                      # el peso publicado, elegido por ACIERTO

    # Los tres optimos sobre el bloque de SELECCION, que son tres pesos distintos: ahi
    # esta el hallazgo de metodo. El RPS pide no mezclar; el acierto y el ROI no.
    s = cur[(cur["rival"] == "juego") & (cur["bloque"] == "seleccion")]
    print("optimos sobre el bloque de seleccion (rival = las 59 de juego):")
    print(f"   por RPS     : w={s.loc[s['rps'].idxmin(), 'w']:.2f}")
    print(f"   por acierto : w={s.loc[s['acierto'].idxmax(), 'w']:.2f}   <- el elegido")
    print(f"   por ROI     : w={s.loc[s['roi'].idxmax(), 'w']:.2f}")
    print("   -> el ROI sobre 1140 partidos tiene un error tipico de ~3 pp: elegir con "
          "el es elegir ruido")

    # Lo que decide: el signo de FUERA. `d_ok` es acierto, `d_ben` ROI, `d_rps` RPS, y
    # el signo de d_rps esta invertido por construccion (menos RPS es mejor).
    p = (par[(par["rival"] == "juego") & (par["w"] == W_PUB)]
         .set_index("bloque")
         .reindex(["seleccion", "medicion", "confirmacion"]))
    print(f"\ndiferencia emparejada contra la semilla, w={W_PUB}:")
    for b, r in p.iterrows():
        marca = "(eligio)" if b == "seleccion" else "(NO eligio)"
        print(f"   {b:<13} n={int(r['n']):>4} {marca:<12} "
              f"acierto {r['d_ok'] * 100:+.2f} pp "
              f"[{r['lo_ok'] * 100:+.2f},{r['hi_ok'] * 100:+.2f}]   "
              f"ROI {r['d_ben'] * 100:+.2f} pp "
              f"[{r['lo_ben'] * 100:+.2f},{r['hi_ben'] * 100:+.2f}]   "
              f"RPS {-r['d_rps']:+.6f}")

    fuera = p.loc[["medicion", "confirmacion"]]
    repite_ok = bool((fuera["d_ok"] > 0).all())
    repite_roi = bool((fuera["d_ben"] > 0).all())
    excluye = bool(((fuera["lo_ok"] > 0) | (fuera["lo_ben"] > 0)).any())
    print(f"\n   el signo repite fuera: acierto {repite_ok} · ROI {repite_roi}"
          f"   <- es la UNICA de las nueve rutas que lo hace")
    print(f"   alguna diferencia de fuera excluye el cero: {excluye}"
          f"   <- el signo no se desmorona, que no es lo mismo que estar demostrado")
    print(f"   el RPS empeora en los tres bloques: "
          f"{bool((p['d_rps'] > 0).all())}   <- es el precio declarado de la decision")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.0))

    # Izquierda: la curva de RPS por bloque. El minimo esta en w=1 (no mezclar) en los
    # tres, que es por que la mezcla necesitaba que se levantara la regla 4.
    for b, est in (("seleccion", ":"), ("medicion", "-"), ("confirmacion", "--")):
        t = cur[(cur["rival"] == "juego") & (cur["bloque"] == b)].sort_values("w")
        ax1.plot(t["w"], t["rps"], est, marker="o", ms=3, label=b)
    ax1.axvline(W_PUB, color=PALETA["modelo"], lw=1.4,
                label=f"w={W_PUB} (publicado)")
    ax1.set_xlabel("w · peso del mercado")
    ax1.set_ylabel("RPS (menos es mejor)")
    ax1.set_title("El RPS pide no mezclar, en los tres bloques")
    ax1.legend(fontsize=8)

    # Derecha: el acierto, que es con lo que se eligio el peso, y donde el optimo si es
    # interior y cae en el mismo sitio dentro y fuera.
    for b, est in (("seleccion", ":"), ("medicion", "-"), ("confirmacion", "--")):
        t = cur[(cur["rival"] == "juego") & (cur["bloque"] == b)].sort_values("w")
        ax2.plot(t["w"], t["acierto"] * 100, est, marker="o", ms=3, label=b)
    ax2.axvline(W_PUB, color=PALETA["modelo"], lw=1.4)
    ax2.set_xlabel("w · peso del mercado")
    ax2.set_ylabel("acierto (%)")
    ax2.set_title("El acierto si tiene optimo interior, y cae donde fuera tambien")
    ax2.legend(fontsize=8)

    fig.tight_layout()
    plt.show()

### C.9.5 · La décima ruta: partir el censo, y el mayor salto del proyecto

Las nueve rutas anteriores preguntan por el **RPS**, que es la métrica con la que se
construyó el banco. El 20-oct-2026 Manuel cambió el criterio —«el modelo que voy a
utilizar va a ser el que mejore el accuracy y el ROI al máximo… olvídate del RPS para
encontrar el mejor modelo»— y esta es la primera ruta que lo obedece.

En su forma literal la pregunta es «maximiza el acierto y el ROI sobre todas las
temporadas disponibles», y contestada así se contesta mal: la tabla que sale enumera en
su campo `temporadas` las seis mismas sobre las que reporta, de modo que **elige y mide
en los mismos 2280 partidos**. Sus dos ganadores —55.00 % de acierto con nueve variables
y +4.56 % de ROI con diez— son incitables por la regla 5, y el optimismo crece con el
número de configuraciones probadas, que allí fueron miles.

La única forma de cerrar ese hueco es **partir el censo**: elegir en `seleccion` (1140
partidos) y medir en `medicion` (1140) y en `confirmacion` (760, con el arranque abierto
a 2016/17 sola, como hace `seleccion.confirma()`). Ninguno de los 1900 de fuera
participó en elegir nada y los 760 son anteriores a todo, así que la regla 6 queda en
pie. Lo que esto permite y la pasada literal no: si un conjunto bate a la semilla en
acierto y en ROI **en las dos ventanas de fuera**, su cifra es citable y es el modelo del
TFM.

**Dentro del bloque que eligió, el salto es el mayor medido en todo el proyecto.** El
ganador por acierto —siete variables, `C=0.3, L2`— sube de 52.46 % a **54.30 %** y de
+2.20 % a **+9.50 %** de ROI a la mejor cuota. Visto solo, parece el hallazgo del
trabajo.

**Fuera se parte en dos**, que es la firma de haber escogido el máximo de una nube:

| ventana | n | acierto contra la semilla | ROI máx contra la semilla |
|---|---|---|---|
| `medicion` (no eligió) | 1140 | +0.09 pp [−1.23, +1.41] | +1.55 pp [−2.32, +5.42] |
| `confirmacion` (no eligió) | 760 | **−1.05 pp** [−2.64, +0.54] | **−1.82 pp** [−6.19, +2.54] |
| las dos juntas | 1900 | **−0.37 pp** [−1.38, +0.65] | +0.20 pp [−2.71, +3.11] |

Gana en una ventana y pierde en la otra, y sobre los 1900 juntos el acierto queda por
debajo de la semilla. **El signo no repite: se descarta, y es la décima ruta nula.**

Y hay un daño que sí es significativo, el único de esta sección: **el RPS empeora en las
tres ventanas con los tres intervalos excluyendo el cero** (+0.001500 en `medicion`,
+0.002828 en `confirmacion`, +0.002031 en las dos juntas). Comparado con la mezcla de
C.9.4 —que pierde 0.000527 de RPS sin significación y cuyo ROI sí repite signo fuera—,
este conjunto de siete es **estrictamente peor en las dos cosas a la vez**.

La pista estaba en su propia estructura, y sirve de detector para la próxima vez: de las
tres columnas de precio retiene **solo `mkt_p_away`**. Tirar dos tercios de lo único que
el proyecto ha medido como señal fiable, para ganar acierto justo en el bloque donde se
elige, es lo que hace un conjunto ajustado al bloque.

> Las cifras de dentro de `seleccion` no son citables (regla 5); las de `medicion` y
> `confirmacion` sí, y son las que descartan la ruta. Lo producen `_maximo.py` y
> `_censo.py`, y lo dejan en `maximo*.csv` y `censo*.csv` bajo `outputs/`, que no se
> versiona. Las rutas las declara `seleccion.py`, para que este cuaderno no las lleve
> escritas a mano.

In [ ]:
# Regla 8: las dos tablas se leen de disco. Las produjo `_censo.py` —la misma busqueda de
# acierto y ROI de `_maximo.py`, pero con el censo PARTIDO: elige en `seleccion` y mide en
# `medicion` y en `confirmacion`— y viven bajo `outputs/`, que no se versiona.
import json

if not (sl.CENSO_GANADORES.exists() and sl.CENSO_FUERA.exists()):
    print(f"sin la decima ruta en {sl.CENSO_CURVA}")
else:
    gan = json.loads(sl.CENSO_GANADORES.read_text(encoding="utf-8"))
    fue = pd.read_csv(sl.CENSO_FUERA)
    sem = gan["semilla"]

    # DENTRO del bloque que eligio: el mayor salto medido en el proyecto, y por eso
    # mismo el mas sospechoso. Son 2314 evaluaciones y el optimismo crece con ellas.
    print(f"{gan['evaluaciones']:,} evaluaciones en {gan['minutos']:.0f} min, "
          f"eligiendo con '{gan['elige']}' sobre un banco de {gan['n_banco']}\n")
    print(f"--- dentro de '{gan['elige']}' ({sem['n']:,} partidos) ---")
    print(f"   semilla          acierto {sem['acierto'] * 100:.2f}%  "
          f"ROI max {sem['roi_max'] * 100:+.2f}%  RPS {sem['rps']:.6f}")
    for cual in ("ganador_acierto", "ganador_roi"):
        g = gan[cual]
        print(f"   {cual:<16} acierto {g['acierto'] * 100:.2f}%  "
              f"ROI max {g['roi_max'] * 100:+.2f}%  RPS {g['rps']:.6f}  "
              f"(k={len(g['conjunto'])})")
    print("   -> el mayor salto del proyecto, y medido donde se eligio: incitable")

    # FUERA se parte en dos, y eso es el veredicto: gana en `medicion` y pierde en
    # `confirmacion`. Un signo que se invierte es el maximo de una nube, no una ventaja.
    print("\n--- fuera, contra la SEMILLA sobre los mismos partidos ---")
    for b in ("medicion", "confirmacion", "ambos"):
        r = fue[(fue["bloque"] == b) & (fue["quien"] == "ganador_acierto")].iloc[0]
        print(f"   {b:<13} n={int(r['n']):>4}  acierto {r['dac_semilla'] * 100:+.2f} pp "
              f"[{r['inf_dac_semilla'] * 100:+.2f},{r['sup_dac_semilla'] * 100:+.2f}]"
              f"   ROI max {r['droimax_semilla'] * 100:+.2f} pp "
              f"[{r['inf_droimax_semilla'] * 100:+.2f},"
              f"{r['sup_droimax_semilla'] * 100:+.2f}]")
    amb = fue[(fue["bloque"] == "ambos") & (fue["quien"] == "ganador_acierto")].iloc[0]
    med = fue[(fue["bloque"] == "medicion") & (fue["quien"] == "ganador_acierto")].iloc[0]
    con = fue[(fue["bloque"] == "confirmacion")
              & (fue["quien"] == "ganador_acierto")].iloc[0]
    print(f"\n   el signo del acierto repite fuera: "
          f"{bool(med['dac_semilla'] > 0) == bool(con['dac_semilla'] > 0)}"
          "   <- se parte en dos, asi que NO")

    # Y el dano que si es significativo: el RPS empeora en las tres ventanas con los
    # tres intervalos excluyendo el cero. Esto no es ruido de muestra.
    print("\n--- el RPS, que es lo unico que sale significativo ---")
    rps = fue[fue["quien"] == "ganador_acierto"]
    for _, r in rps.iterrows():
        print(f"   {r['bloque']:<13} RPS {r['drps']:+.6f} "
              f"[{r['inf_drps']:+.6f},{r['sup_drps']:+.6f}]  excluye el cero: "
              f"{bool(r['inf_drps'] > 0)}")
    print("   -> frente a la mezcla de C.9.4, que pierde 0.000527 de RPS SIN "
          "significacion y cuyo\n      ROI si repite signo fuera, este conjunto es peor "
          "en las dos cosas a la vez")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

    # Izquierda: el ROI maximo por bloque. La barra de dentro es enorme y las dos de
    # fuera se cancelan; es la figura entera del argumento.
    BLO = ["medicion", "confirmacion", "ambos"]
    x = range(len(BLO))
    for i, (quien, col) in enumerate((("semilla", PALETA["apoyo"]),
                                      ("ganador_acierto", PALETA["modelo"]))):
        t = fue[fue["quien"] == quien].set_index("bloque").reindex(BLO)
        ax1.bar([v + i * 0.38 for v in x], t["roi_max"] * 100, 0.38,
                color=col, label=quien)
    ax1.axhline(0, color="black", lw=0.8)
    ax1.set_xticks([v + 0.19 for v in x])
    ax1.set_xticklabels(BLO)
    ax1.set_ylabel("ROI a la mejor cuota (%)")
    ax1.set_title("Fuera se parte en dos: gana en una ventana y pierde en la otra")
    ax1.legend(fontsize=8)

    # Derecha: el RPS emparejado con su intervalo. Las tres barras caen enteras a la
    # derecha del cero, que es lo que no hace ninguna otra cifra de esta ruta.
    t = rps.set_index("bloque").reindex(BLO)
    ax2.errorbar(t["drps"], range(len(t)),
                 xerr=[t["drps"] - t["inf_drps"], t["sup_drps"] - t["drps"]],
                 fmt="o", color=PALETA["mercado"], capsize=4)
    ax2.axvline(0, color="black", lw=1.2)
    ax2.set_yticks(range(len(t)))
    ax2.set_yticklabels(BLO)
    ax2.set_xlabel("RPS contra la semilla (mas es PEOR)")
    ax2.set_title("El dano en RPS si excluye el cero, en las tres ventanas")

    fig.tight_layout()
    plt.show()

### C.9.6 · La undécima ruta: elegir un procedimiento y no un conjunto

Las diez rutas anteriores comparten una limitación de cobertura: todas eligen con
2019/20–2021/22 y puntúan fuera, de modo que **dejan tres temporadas sin cifra**. Manuel
pidió «el mejor modelo que mejore las métricas de ROI y accuracy en todo el histórico de
partidos desde 17/18 hasta hoy sin excepción», y las dos mitades de esa petición se
estorban: ampliar el bloque de elección arreglaría la cobertura y rompería la regla 5.

Solo son compatibles **eligiendo un procedimiento y no un conjunto**. Para cada temporada
S se repite la búsqueda entera **con solo partidos anteriores a S**, y con lo que salga se
predice S. El conjunto cambia con el origen —eso es el punto, no un defecto—; lo que se
fija y se juzga es el procedimiento. Son nueve orígenes por dos criterios de búsqueda
—`accuracy` y `ventaja`, los dos que la regla 4 ampliada deja arbitrar—, 4.2 h de
cómputo.

Un detalle del arranque que conviene saber antes de leer la tabla: los orígenes de
2017/18 y 2018/19 tienen 0 y 380 partidos de pasado, por debajo del suelo declarado de
760, así que **esas dos temporadas se predicen con la semilla** en las dos pistas. No es
una concesión: es lo que significa elegir con el pasado cuando todavía no hay pasado.

Cuatro pistas sobre el mismo censo, con el listón al lado. Las cifras son las del **censo
limpio** —las seis temporadas que no eligieron el peso de la mezcla publicada, 1978
partidos, el único censo donde las cuatro columnas se comparan—:

| pista | acierto | RPS | ROI b365 | ventaja sobre el listón |
|---|---|---|---|---|
| listón (favorito de b365) | 53.54 % | — | −2.78 % | — |
| semilla (3 de precio) | 53.44 % | 0.193962 | −2.97 % | −0.19 pp [−1.53, +1.20] |
| publicado (mezcla, 62) | 53.84 % | 0.194437 | −1.60 % | **+1.18 pp** [−0.84, +3.33] |
| rodante por acierto | 53.49 % | 0.194118 | −2.64 % | +0.14 pp [−1.64, +1.93] |
| rodante por ventaja | 53.59 % | 0.194335 | −1.96 % | +0.82 pp [−1.35, +3.01] |

**Ni un intervalo excluye el cero, ninguno de los cuatro**, y sobre los 3118 enteros
tampoco: el mayor es el de la mezcla publicada, +1.52 pp [−0.19, +3.31]. De las doce
comparaciones emparejadas entre pistas —seis pares en los dos censos— **solo una excluye
el cero**, `publicado` contra `semilla` sobre los 3118, y en el censo limpio ya no.

Esa es la respuesta honesta a lo que se pidió: **cubiertas las nueve temporadas sin
excepción, ninguna ruta del proyecto produce una ventaja distinguible de cero contra el
favorito de Bet365, y la mejor de las cuatro sigue siendo la mezcla que ya se publica.**
Lo que esta ruta aporta no es un modelo mejor —pierde, y además no hay nada que ganar—:
es la primera cifra del trabajo que **cubre las nueve temporadas sin que ninguna haya
participado en elegir lo que la predice**, y por tanto la primera citable entera.

Tres cosas del cuadro que no hay que volver a medir, y la primera es una predicción
fallada:

- **Esperaba que la pista de ventaja quedara por debajo de la de acierto, y queda por
  encima.** El argumento era que su tolerancia es dos o tres veces más ancha —0.0068
  contra 0.0025 en el origen de 2020/21— y compra ruido. Adopta más profundo, sí (hasta
  nueve variables en 2022/23, frente a las cuatro de la otra pista), pero da +0.82 pp
  contra +0.14 y más acierto. **Lo que cuesta está en el RPS**: 0.194335 contra 0.194118,
  0.000217 peor, que es exactamente lo que la regla 4 obliga a seguir reportando aunque
  ya no arbitre.
- **Las dos pistas rodantes difieren en 91 partidos de 3118**, el 2.9 %; emparejadas, la
  diferencia es +0.36 pp [−1.09, +1.79]. Las cuatro coinciden en el signo en más del
  95 % de los partidos, así que las cifras marginales de la tabla **exageran las
  distancias**, y la comparación que vale es la emparejada.
- **La celda más bonita es la única incitable**: la pista de acierto en 2026/27 da
  +6.18 % de ROI y +8.21 pp de ventaja, y son 78 partidos con un error típico de
  10.93 pp. Está a **0.75 errores típicos de cero**.

Y un hallazgo lateral que va contra la directriz que lo produjo: **el orden de las
candidatas por RPS es más estable entre orígenes que el orden por acierto**, y la brecha
**se abre con la distancia**, que es lo que lo hace un hallazgo y no una curiosidad. Rho
de Spearman 0.78 contra 0.69 promediando los 21 pares del criterio de acierto, y **0.588
contra 0.371** en el par más lejano, 2019/20 contra 2026/27. No cambia quién elige, que
lo fijó Manuel, pero es el mecanismo por el que una búsqueda guiada por acierto compra
daño en RPS sin anunciarlo.

> Lo producen `_rodante.py` y `_estabilidad.py`, y lo dejan en `rodante_*.csv/json` y
> `estabilidad*.csv` bajo `outputs/`, que no se versiona. Las rutas las declara
> `seleccion.py`.

In [ ]:
# Regla 8: la tabla se lee de disco. La produjo `_rodante.py` —nueve origenes por dos
# criterios, cada temporada predicha con un conjunto elegido con solo su pasado— y vive
# bajo `outputs/`, que no se versiona.
import json

if not (sl.RODANTE_TEMPORADA.exists() and sl.RODANTE_RESUMEN.exists()):
    print(f"sin la ruta rodante en {sl.RODANTE}")
else:
    tmp = pd.read_csv(sl.RODANTE_TEMPORADA)
    res = json.loads(sl.RODANTE_RESUMEN.read_text(encoding="utf-8"))
    PISTAS = ["semilla", "publicado", "rodante_ac", "rodante_roi"]

    # El censo LIMPIO son las seis temporadas que no eligieron el peso de la mezcla, y
    # es el unico donde las cuatro pistas se comparan: `publicado` vio las otras tres.
    print(f"censo entero: {res['n_partidos']:,} partidos en {res['pliegues']} pliegues "
          f"de {len(res['origenes'])} temporadas")
    print(f"censo limpio: {res['limpio']['semilla']['n']:,} partidos "
          f"(sin {', '.join(res['sucias_peso'])}, que eligieron el peso)\n")

    fil = []
    for cual, etiq in (("limpio", "censo limpio"), ("agregado", "las nueve")):
        for p in PISTAS:
            d = res[cual][p]
            fil.append({"censo": etiq, "pista": p, "n": d["n"],
                        "acierto": d["acierto"], "rps": d["rps"],
                        "roi": d["roi_b365"], "liston": d["roi_ref_b365"],
                        "vent": d["vent_b365"],
                        "inf": d["vent_b365_inf"], "sup": d["vent_b365_sup"]})
    tab = pd.DataFrame(fil)
    for etiq in ("censo limpio", "las nueve"):
        t = tab[tab["censo"] == etiq]
        ref = t.iloc[0]
        print(f"--- {etiq} ({int(ref['n']):,} partidos) · liston "
              f"{ref['liston'] * 100:+.2f}% ---")
        for _, r in t.iterrows():
            print(f"   {r['pista']:<12} acierto {r['acierto'] * 100:.2f}%  "
                  f"RPS {r['rps']:.6f}  ROI {r['roi'] * 100:+.2f}%  "
                  f"ventaja {r['vent'] * 100:+.2f} pp "
                  f"[{r['inf'] * 100:+.2f},{r['sup'] * 100:+.2f}]")

    # Lo que decide: ningun intervalo excluye el cero. Que una pista gane por nivel no
    # la hace distinguible del liston, y el intervalo es lo que hay que mirar. La
    # comprobacion va FILA A FILA: un intervalo excluye el cero cuando sus dos extremos
    # caen al mismo lado, y agregar los extremos por separado no contesta eso.
    cruza = (tab["inf"] <= 0) & (tab["sup"] >= 0)
    mejor = tab[tab["censo"] == "censo limpio"].nlargest(1, "vent")["pista"].iloc[0]
    print(f"\n   alguna ventaja excluye el cero: {not bool(cruza.all())}")
    print(f"   la mejor del censo limpio es '{mejor}' <- es la que YA se publica, "
          "asi que no hay nada que adoptar")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

    # Izquierda: la ventaja sobre el liston temporada a temporada. El vaiven es el
    # argumento: lo que la media agregada esconde.
    por = tmp[tmp["temporada"] != "TODAS"]
    for p, est in zip(PISTAS, (":", "-", "--", "-.")):
        t = por[por["pista"] == p].sort_values("temporada")
        ax1.plot(t["temporada"], t["vent_b365"] * 100, est, marker="o", ms=4, label=p)
    ax1.axhline(0, color=PALETA["apoyo"], lw=1.0)
    ax1.set_ylabel("ventaja sobre el liston (pp)")
    ax1.set_title("Temporada a temporada: el vaiven que la media esconde")
    ax1.tick_params(axis="x", rotation=60)
    ax1.legend(fontsize=8)

    # Derecha: nivel contra intervalo en el censo limpio. Lo que hay que ver es que las
    # cuatro barras de error cruzan el cero.
    t = tab[tab["censo"] == "censo limpio"]
    ax2.errorbar(t["vent"] * 100, range(len(t)),
                 xerr=[(t["vent"] - t["inf"]) * 100, (t["sup"] - t["vent"]) * 100],
                 fmt="o", color=PALETA["modelo"], capsize=4)
    ax2.axvline(0, color=PALETA["mercado"], lw=1.4)
    ax2.set_yticks(range(len(t)))
    ax2.set_yticklabels(t["pista"])
    ax2.set_xlabel("ventaja sobre el liston (pp)")
    ax2.set_title("Censo limpio: las cuatro cruzan el cero")

    fig.tight_layout()
    plt.show()

### C.9.7 · El máximo por ROI y acierto sobre el censo de nueve

Con el censo ya fijado —2017/18–2024/25 y 2026/27, **3118 partidos en 345 pliegues**— se
puede hacer la pregunta en su forma literal: «el mejor modelo en términos de roi y
accuracy… que maximice esas métricas tanto en cómputo global como a nivel de temporada».
Esta pasada **no es una ruta más: no busca nada, puntúa.** Mide los **50 conjuntos
distintos** que han producido las once rutas anteriores sobre los **mismos** 3118
partidos, temporada a temporada.

El resultado es que **no hay conjunto que adoptar**, y las tres lecturas lo dicen con
tres fuerzas distintas.

**Lectura 1, el global.** El máximo lo firma `maximo_roi` (k=10): acierto 54.11 %, ROI
+1.35 %, ventaja **+4.22 pp [+1.72, +6.77]** sobre el listón de los 3118 (−2.87 % y
53.11 %). Ese intervalo excluye el cero, y **es exactamente por eso por lo que no es
citable**: ese conjunto se buscó sobre partidos que están dentro de estos 3118, así que
la cifra arrastra el defecto que C.9.5 diagnosticó en vez de repararlo. El publicado de
62 queda en 53.50 % y +1.52 pp [−0.19, +3.31].

Del mismo cuadro sale un aviso que hay que leer con cuidado, porque **meter el banco
entero cuesta acierto y RPS y no cuesta ventaja**: `todas65` cae a 50.58 % de acierto y
0.202736 de RPS, y `juego59` a 50.03 % y 0.204590, pero sus ventajas sobre el listón
siguen saliendo positivas (+1.66 y +0.25 pp) y la peor de las 50 es `semilla+fuerza` con
−0.93 pp. El ROI plano sobre 3118 partidos tiene un error típico de 1.79 pp y **no
distingue un modelo que acierta el 50 % de uno que acierta el 53.5 %**. El daño existe y
está medido; quien lo mide es el acierto, no el dinero — y de ahí que la directriz pida
dos métricas y haya que seguir reportando las tres.

**Lectura 2, por temporada: no hay respuesta que dar, y eso es un hallazgo y no una
evasiva.** Las nueve temporadas las firman **nueve conjuntos distintos, en los dos
criterios, y ninguno repite**. El puesto medio del ganador en las otras ocho va de 12.2 a
40.6 de 50 por acierto: `escalera+clasificacion` gana 2023/24 con .5605 y promedia .5110
fuera. Nueve máximos sobre 380 partidos con un error típico de 5.0 pp son nueve sorteos.
No existe «el conjunto que maximiza cada temporada», existe el que ganó ahí.

**Lectura 3, dejar una fuera: la única citable, y no excluye el cero.** Se elige con ocho
temporadas y se puntúa en la novena, que no participó en elegir. Guiando por acierto
salen 4 conjuntos distintos y **53.30 % contra 53.11 %** del listón, con ventaja
**+1.46 pp [−0.42, +3.34]**; guiando por ventaja, 3 conjuntos y 53.11 % contra 53.11 %
—el mismo acierto al decimal— con **+1.45 pp [−0.84, +3.69]**. **Los dos quedan por
debajo del +1.52 pp del modelo publicado**, y los vaivenes por vuelta dicen de dónde sale
la media: guiando por ventaja, +8.41 pp en 2019/20 contra −4.55 pp en 2026/27.

Un aviso sobre el censo llamado **limpio**, que son las seis temporadas que no eligieron
el peso de la mezcla: **es censo limpio para el publicado y para la semilla, y no lo es
para ningún conjunto de búsqueda.** `maximo_roi` se buscó sobre 2019/20–2024/25 y pisa
tres de las seis; `rod_ve_202425`, que ahí firma el máximo con +3.61 pp [+1.35, +5.97],
se eligió con el pasado de 2024/25, que contiene cuatro de las seis. Ese +3.61 **no es
una cifra de fuera de muestra**.

Y la voraz con los 3118 **dentro** del bloque de elección —el máximo de información que
el proyecto puede darle— **se agota en dos variables sobre la semilla**: para a k=5 por
los dos criterios, y en la tolerancia y no en el techo. Por acierto, la mejor candidata
del paso 3 (`form_ga_ewm_diff`) mejoraba +0.000000 contra una tolerancia de 0.00153093.

> La lectura citable es la 3; las lecturas 1 y 2 no lo son (regla 5) y la celda lo
> imprime en su propia salida, no solo aquí arriba. Lo produce `_nueve.py` y lo deja en
> `nueve_*.csv/json` bajo `outputs/`, que no se versiona.

In [ ]:
# Regla 8: las tres lecturas se leen de disco. Las produjo `_nueve.py`, que no busca
# nada: puntua los 50 conjuntos que han producido las once rutas sobre los MISMOS
# partidos del censo, temporada a temporada.
import json

if not (sl.NUEVE_ACERVO.exists() and sl.NUEVE_LECTURA.exists()):
    print(f"sin el acervo del censo de nueve en {sl.NUEVE}")
else:
    ace = pd.read_csv(sl.NUEVE_ACERVO)
    lec = json.loads(sl.NUEVE_LECTURA.read_text(encoding="utf-8"))
    print(f"{lec['n_conjuntos']} conjuntos distintos sobre {lec['n_partidos']:,} "
          "partidos\n")

    # LECTURA 1, el agregado. El maximo excluye el cero, y es por eso mismo por lo que
    # no es citable: el conjunto se busco sobre partidos que estan dentro del censo.
    tod = ace[ace["temporada"] == "TODAS"].copy()
    liston = tod["roi_ref_b365"].iloc[0]
    print(f"--- lectura 1: el agregado (liston {liston * 100:+.2f}%, "
          f"acierto {tod['acierto_ref'].iloc[0] * 100:.2f}%) ---")
    for cual, col in (("ventaja", "vent_b365"), ("acierto", "acierto")):
        g = tod.nlargest(1, col).iloc[0]
        print(f"   maximo por {cual:<8}: {g['conjunto']:<22} k={int(g['k']):<3} "
              f"acierto {g['acierto'] * 100:.2f}%  RPS {g['rps']:.6f}  "
              f"ventaja {g['vent_b365'] * 100:+.2f} pp "
              f"[{g['vent_b365_inf'] * 100:+.2f},{g['vent_b365_sup'] * 100:+.2f}]")
    for p in ("publicado", "semilla", "todas65", "juego59"):
        r = tod[tod["conjunto"] == p]
        if not r.empty:
            r = r.iloc[0]
            print(f"   {p:<20} k={int(r['k']):<3} acierto {r['acierto'] * 100:.2f}%  "
                  f"RPS {r['rps']:.6f}  ventaja {r['vent_b365'] * 100:+.2f} pp "
                  f"[{r['vent_b365_inf'] * 100:+.2f},{r['vent_b365_sup'] * 100:+.2f}]")
    print("   -> meter el banco entero cuesta acierto y RPS y NO cuesta ventaja: el "
          "ROI plano\n      sobre 3118 partidos no distingue el 50% del 53.5%")

    # LECTURA 2: por temporada no hay respuesta que dar, y eso es el hallazgo.
    pt2 = lec["por_temporada"]["acierto"]
    print(f"\n--- lectura 2: por temporada ({pt2['conjuntos_distintos']} conjuntos "
          "distintos en nueve temporadas, ninguno repite) ---")
    for s, d in list(pt2["fuera"].items())[:3]:
        print(f"   {s}: gana '{d['conjunto']}' con {d['en_la_suya'] * 100:.2f}% y "
              f"promedia {d['media_otras'] * 100:.2f}% fuera "
              f"(puesto medio {d['puesto_medio_otras']:.1f} de 50)")
    print("   -> nueve maximos sobre 380 partidos con un error tipico de 5.0 pp son "
          "nueve sorteos")

    # LECTURA 3, la UNICA citable: se elige con ocho temporadas y se puntua en la
    # novena, que no participo en elegir.
    print("\n--- lectura 3: dejar una fuera (la unica citable) ---")
    for cri in ("acierto", "vent_b365"):
        d = lec["dejar_una_fuera"][cri]
        v = d["agregado"]
        print(f"   guiado por {cri:<10} acierto {v['acierto'] * 100:.2f}% contra "
              f"{v['acierto_ref'] * 100:.2f}% del liston  ·  ventaja "
              f"{v['vent_b365'] * 100:+.2f} pp "
              f"[{v['vent_b365_inf'] * 100:+.2f},{v['vent_b365_sup'] * 100:+.2f}]")
    pub = tod[tod["conjunto"] == "publicado"]["vent_b365"]
    if not pub.empty:
        print(f"   -> las dos quedan por debajo del {pub.iloc[0] * 100:+.2f} pp del "
              "modelo publicado, y ninguna excluye el cero")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

    # Izquierda: la nube de los 50. El eje x es el acierto y el y la ventaja, y lo que
    # se ve es que no hay frontera: la ventaja no ordena por acierto.
    ax1.scatter(tod["acierto"] * 100, tod["vent_b365"] * 100, s=18,
                color=PALETA["apoyo"], label="los 50 conjuntos")
    for p, col in (("publicado", PALETA["modelo"]), ("semilla", PALETA["mercado"])):
        r = tod[tod["conjunto"] == p]
        if not r.empty:
            ax1.scatter(r["acierto"] * 100, r["vent_b365"] * 100, s=90, color=col,
                        zorder=3, label=p)
    ax1.axhline(0, color=PALETA["apoyo"], lw=1.0)
    ax1.set_xlabel("acierto (%)")
    ax1.set_ylabel("ventaja sobre el liston (pp)")
    ax1.set_title("Los 50 conjuntos sobre el censo de nueve")
    ax1.legend(fontsize=8)

    # Derecha: el ganador de cada temporada y su puesto medio en las otras ocho. Una
    # barra alta es un conjunto que gano donde se le midio y es del monton fuera.
    pm = pd.DataFrame([
        {"temporada": s, "puesto": d["puesto_medio_otras"]}
        for s, d in pt2["fuera"].items()])
    ax2.bar(pm["temporada"], pm["puesto"], color=PALETA["modelo"])
    ax2.axhline(25.5, color=PALETA["mercado"], lw=1.4, label="el monton (25.5 de 50)")
    ax2.set_ylabel("puesto medio en las otras ocho")
    ax2.set_title("El ganador de cada temporada, fuera de su temporada")
    ax2.tick_params(axis="x", rotation=60)
    ax2.legend(fontsize=8)

    fig.tight_layout()
    plt.show()

## C.10 · La cifra citable

Una sola llamada, sobre el bloque de **medición**, con el conjunto ya cerrado. La tabla
que sale es la que va a la memoria: temporada a temporada, el acierto y el rendimiento
del modelo frente al favorito de Bet365 sobre exactamente los mismos partidos.

Que se llame una vez es una restricción del método, no del código. Nada impide ejecutar
la celda con veinte conjuntos distintos; lo que ocurre si se hace es que el bloque de
medición se convierte en un segundo bloque de selección y la cifra deja de significar lo
que dice significar. Si hace falta volver a elegir, se vuelve a C.8 — no aquí.

Los dos aciertos de la tabla no son el mismo número medido dos veces: `acierto` es el
del modelo sobre **todas** las jornadas de la temporada, y `acierto_apostados` /
`acierto_b365` son los de cada apostante sobre los partidos en los que apostó, que no
tienen por qué ser los mismos ni ser los mismos en número.

El holdout 2025/26 sigue sellado. Se abre en la Parte E, una sola vez, con el conjunto y
la familia definitivos.

> **Esto no es la rutina semanal.** C.8–C.10 se ejecutan una vez por trabajo, no cada
> jornada. La emisión semanal parte del conjunto ya guardado, que `build_models.py`
> recoge con `seleccion.carga()` sin que nadie lo copie a mano en ningún sitio.


In [ ]:
# ---- EDITABLE -------------------------------------------------------------
# Por defecto se mide lo que C.9 eligió **después** de la cuarta puerta: `flotante`
# revierte `sel.conjunto` a la semilla si la confirmación rechaza, de modo que aquí no
# se puede medir por descuido un conjunto que C.9.1 haya devuelto. Lo que la voraz había
# propuesto sigue en `sel.propuesto`, que es lo que hace legible el rechazo. Para medir
# la selección guardada en una sesión anterior, sin repetir la búsqueda:
#     CONJUNTO_SEL = list(sl.carga()["conjunto"])
CONJUNTO_SEL = list(sel.conjunto)
# ---------------------------------------------------------------------------

ensayo_sel = sl.mide(CONJUNTO_SEL, FAMILIA_SEL, feat=feat)
display(sl.informe(ensayo_sel))

---
# Parte D · Lectura completa de un ensayo

La Parte C produce cifras de titular: acierto, RPS, ventaja. Esta parte las abre.
Se aplica a **un** ensayo —el que se tenga en la variable `ensayo`— y responde a las
preguntas que un tribunal hará: si el rendimiento se sostiene en el tiempo, si las
probabilidades significan lo que dicen, dónde se equivoca, si la ventaja nace de
discrepar con el precio o de copiarlo, cómo se comportó el banco y qué está mirando
el modelo.

Todas las lecturas parten de `ensayo.pred`, que ya trae las cuotas enganchadas.

In [ ]:
# ---- EDITABLE -------------------------------------------------------------
E = ensayo          # el ensayo que se quiere abrir
# ---------------------------------------------------------------------------

pub = E.pred
y   = ev._objetivo(pub)
P   = ev._terna(pub, ev.PROB_COLS)
m   = np.isfinite(P).all(axis=1) & np.isfinite(y)
y, P = y[m], P[m]

print(E.resumen())
print(f"\npartidos evaluados: {len(y):,}   temporadas: {pub['season'].nunique()}")
print(f"cuotas usadas     : {pub.attrs.get('cuotas')}")

## D.1 · Estabilidad temporal

La primera pregunta no es cuánto acierta, sino si acierta **siempre**. Un RPS agregado
excelente que descansa en una temporada afortunada no es un resultado publicable.

El panel compara el RPS del modelo, temporada a temporada, con el de los precios
disponibles en el artefacto. La distancia se mide contra el precio **más exigente** de
cada temporada —el de menor RPS—, y no contra uno fijo: elegir el rival año a año en
función del resultado sería justamente lo que se quiere evitar, y tomar siempre el más
débil regalaría una ventaja que no existe.

In [ ]:
mod = E.por_temporada.reset_index()
series = {}
for nombre_ref in ("consenso", "b365", "pinnacle"):
    Q = ev.reference_probs(pub, nombre_ref)
    if Q is None:
        continue
    filas = []
    for temporada, sub in pub.groupby("season", sort=True):
        idx = sub.index
        Qi = ev.reference_probs(sub, nombre_ref)
        yi = ev._objetivo(sub)
        ok = np.isfinite(Qi).all(axis=1) & np.isfinite(yi)
        if not ok.any():
            continue
        filas.append({"season": temporada,
                      "rps": float(ev.rps_vector(yi[ok], Qi[ok]).mean())})
    if filas:
        series[nombre_ref] = pd.DataFrame(filas).set_index("season")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
x = np.arange(len(mod))
ax[0].plot(x, mod["rps"], marker="o", lw=1.8, color=PALETA["modelo"], label=E.nombre)
for nombre_ref, s in series.items():
    ax[0].plot(x, s.reindex(mod["season"])["rps"].to_numpy(), marker="s", ms=4,
               lw=1.2, ls="--", label=nombre_ref)
ax[0].set_xticks(x); ax[0].set_xticklabels(mod["season"], rotation=45, ha="right")
ax[0].set_ylabel("RPS (menor es mejor)")
ax[0].set_title("RPS por temporada, modelo y precios")
ax[0].legend(fontsize=8)

if series:
    rps_precios = pd.DataFrame(
        {n: s.reindex(mod["season"])["rps"].to_numpy() for n, s in series.items()})
    mejor_v = rps_precios.min(axis=1).to_numpy()
    mejor_n = rps_precios.idxmin(axis=1).to_numpy()
    dif = mejor_v - mod["rps"].to_numpy()          # >0 = gana el modelo
    ax[1].bar(x, dif, color=np.where(dif > 0, PALETA["modelo"], PALETA["mercado"]))
    ax[1].axhline(0, color="black", lw=1)
    ax[1].set_xticks(x)
    ax[1].set_xticklabels([f"{t}\n{n}" for t, n in zip(mod["season"], mejor_n)],
                          rotation=45, ha="right", fontsize=7)
    ax[1].set_ylabel("RPS mejor precio − RPS modelo")
    ax[1].set_title("Ventaja sobre el precio más exigente de cada temporada")
plt.tight_layout(); plt.show()

if series:
    resumen_rps = pd.DataFrame(
        {n: (s.reindex(mod["season"])["rps"].to_numpy() - mod["rps"].to_numpy())
         for n, s in series.items()},
        index=mod["season"].to_numpy())
    resumen_rps.index.name = "season"
    display(resumen_rps.round(4))
    for nombre_ref in resumen_rps.columns:
        gana = int((resumen_rps[nombre_ref] > 0).sum())
        print(f"  bate a '{nombre_ref}' en {gana} de {len(resumen_rps)} temporadas")

## D.2 · Calibración — si dice 0,60, ¿ocurre el 60 % de las veces?

La calibración es la propiedad que hace que una probabilidad sirva para decidir. Un
modelo perfectamente ordenado pero mal calibrado ordena bien los partidos y a la vez
sobreestima sistemáticamente: si se apuesta contra un precio, esa sobreestimación se
convierte en pérdida real aunque el orden sea correcto.

El diagrama de fiabilidad agrupa las predicciones por nivel declarado y compara con la
frecuencia observada; la diagonal es la calibración perfecta. El **ECE** resume la
distancia media ponderando cada tramo por cuántas predicciones cayeron en él.

La descomposición de Murphy separa lo que el modelo puede arreglar de lo que no:
*fiabilidad* (calibración, corregible), *resolución* (capacidad real de separar
situaciones distintas) e *incertidumbre* (varianza del fenómeno, que es un suelo y no
un defecto).

Aparece un cuarto término, el *residuo*, y conviene entender por qué no sobra. La
identidad se enuncia con tres piezas cuando el pronosticador emite un número finito de
probabilidades distintas: entonces cada tramo contiene un solo valor declarado y el
reparto es exacto. Un modelo continuo no cumple esa condición —dentro de un tramo
conviven probabilidades que solo se parecen— y sustituir cada una por la media del
tramo deja un resto. Ese resto no es error de cálculo: es el precio de agrupar, y
encoge conforme los tramos se afinan.

Con el residuo dentro, la igualdad vuelve a ser exacta y por eso se comprueba: si no
cierra, alguna de las cuatro piezas no mide lo que su nombre dice.

In [ ]:
e    = ev.ece(y, P)
desc = ev.brier_decomposition(y, P)
conf_tab = ev.reliability_table(y, P)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.9))
for k, clase in enumerate(ev.CLASES):
    sub = ev.reliability_table(y, P, clase=k).dropna(subset=["p_media"])
    ax[0].plot(sub["p_media"], sub["frec_obs"], marker="o", ms=4, lw=1.4, label=clase)
ax[0].plot([0, 1], [0, 1], ls="--", lw=1, color="k")
ax[0].set_xlabel("probabilidad declarada"); ax[0].set_ylabel("frecuencia observada")
ax[0].set_title(f"Fiabilidad por clase · ECE = {e:.4f}")
ax[0].legend(fontsize=8)

ax[1].bar(range(len(conf_tab)), conf_tab["n"].to_numpy(), color=PALETA["modelo"])
ax[1].set_xticks(range(len(conf_tab)))
ax[1].set_xticklabels(conf_tab["tramo"], rotation=45, ha="right", fontsize=7)
ax[1].set_xlabel("tramo de confianza"); ax[1].set_ylabel("predicciones")
ax[1].set_title("Cuántas predicciones sostienen cada punto")

partes = {k: desc[k] for k in ("fiabilidad", "resolucion", "incertidumbre",
                               "residuo")}
ax[2].bar(list(partes), list(partes.values()),
          color=[PALETA["mercado"], PALETA["modelo"], PALETA["apoyo"],
                 PALETA["apoyo"]])
for i, (k, v) in enumerate(partes.items()):
    ax[2].text(i, v, f"{v:.4f}", ha="center", va="bottom", fontsize=8)
ax[2].set_title("Brier = fiabilidad − resolución + incertidumbre + residuo")
plt.tight_layout(); plt.show()

print({k: round(float(v), 5) for k, v in desc.items()})
cierre = (desc["fiabilidad"] - desc["resolucion"] + desc["incertidumbre"]
          + desc["residuo"])
print(f"identidad de Murphy: {cierre:.9f} frente a Brier {desc['brier']:.9f}")
# La tolerancia es de coma flotante, no de aproximación: con el residuo dentro
# la igualdad es algebraica y cualquier holgura mayor taparía un error real.
assert abs(cierre - desc["brier"]) < 1e-9, "la descomposición no cierra"
print(f"el residuo pesa un {100 * abs(desc['residuo']) / desc['brier']:.2f} % "
      f"del Brier: es lo que la lectura por tramos no explica")

## D.3 · Matriz de confusión, informe por clase y ROC

Estas tres lecturas responden a la pregunta complementaria: **dónde** se equivoca. Y el
sitio es previsible: el empate.

El empate es la clase minoritaria y, sobre todo, la que casi nunca es la más probable.
Un modelo bien calibrado puede darle 0,28 —que es mucho— y aun así no elegirlo nunca
como clase predicha. Esa cobertura baja **no es un defecto de calibración sino una
consecuencia del argmax**, y es la razón por la que la clase predicha se publica junto
a la terna completa y nunca en su lugar.

El AUC uno-contra-resto mide el orden, que es independiente de ese efecto: dice si el
modelo sabe *en qué partidos* es más probable el empate, aunque nunca llegue a
nombrarlo.

In [ ]:
mc  = ev.confusion_matrix(y, P)
inf_cl = ev.classification_report(y, P)
auc = ev.roc_auc_ovr(y, P)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
M  = mc.to_numpy(dtype="float64")
Mn = M / np.maximum(M.sum(axis=1, keepdims=True), 1)
ax[0].imshow(Mn, cmap="Blues", vmin=0, vmax=1)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax[0].text(j, i, f"{M[i, j]:.0f}\n({Mn[i, j]*100:.0f}%)", ha="center",
                   va="center", fontsize=8,
                   color="white" if Mn[i, j] > .55 else "black")
ax[0].set_xticks(range(len(mc.columns))); ax[0].set_xticklabels(mc.columns)
ax[0].set_yticks(range(len(mc.index)));   ax[0].set_yticklabels(mc.index)
ax[0].set_xlabel("predicho"); ax[0].set_ylabel("observado")
ax[0].set_title("Confusión (recuento y % por fila)")

claves = [k for k in ("auc_home", "auc_draw", "auc_away", "auc_macro", "auc_micro")
          if k in auc]
ax[1].bar(range(len(claves)), [auc[k] for k in claves], color=PALETA["modelo"])
ax[1].axhline(.5, ls="--", lw=1, color=PALETA["mercado"], label="azar")
ax[1].set_xticks(range(len(claves)))
ax[1].set_xticklabels([k.replace("auc_", "") for k in claves], rotation=20, fontsize=8)
ax[1].set_ylim(.4, 1); ax[1].set_ylabel("AUC")
ax[1].set_title("Capacidad de ordenar cada clase"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

display(inf_cl.round(4))
print(f"AUC del empate: {auc.get('auc_draw', float('nan')):.3f} — mide si sabe EN QUÉ "
      "partidos el empate es más probable, aunque no llegue a nombrarlo.")

## D.4 · Desacuerdo con el mercado — dónde vive la ventaja

Si el modelo y el mercado coincidieran en todos los partidos, el modelo sería un
estimador caro del precio. Lo interesante está donde discrepan: ahí es donde una
ventaja puede existir, y también donde el modelo puede estar simplemente equivocado.

La tabla ordena los partidos por magnitud de la discrepancia y mide el rendimiento de
ambos en cada tramo. Un modelo que gana sólo donde coincide con el precio no aporta
nada; uno que gana donde discrepa aporta información que el mercado no incorporaba; uno
que empeora justo donde más discrepa es el patrón clásico del sobreajuste.

In [ ]:
REFERENCIA_DESACUERDO = bt.REFERENCIA      # EDITABLE: "b365" | "consenso" | "pinnacle"

if ev.reference_probs(pub, REFERENCIA_DESACUERDO) is None:
    print(f"el artefacto no trae la referencia {REFERENCIA_DESACUERDO!r}; "
          "no hay lectura de desacuerdo")
    desac = None
else:
    desac = ev.disagreement(pub, referencia=REFERENCIA_DESACUERDO)
    display(desac.round(4))

In [ ]:
if desac is not None and len(desac):
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    x = np.arange(len(desac))
    ax[0].bar(x - .2, desac["rps_modelo"], width=.4, label=E.nombre,
              color=PALETA["modelo"])
    ax[0].bar(x + .2, desac["rps_mercado"], width=.4, label=REFERENCIA_DESACUERDO,
              color=PALETA["mercado"])
    ax[0].set_xticks(x)
    ax[0].set_xticklabels([f"g{int(g)}\nd={d:.3f}\nn={int(n)}" for g, d, n in
                           zip(desac["grupo"], desac["dist_media"], desac["n"])],
                          fontsize=7)
    ax[0].set_xlabel("quintil de desacuerdo"); ax[0].set_ylabel("RPS")
    ax[0].set_title("Donde discrepan, quién acierta"); ax[0].legend(fontsize=8)

    ax[1].bar(x, desac["skill_rps"],
              color=np.where(desac["skill_rps"] > 0, PALETA["modelo"], PALETA["mercado"]))
    ax[1].axhline(0, color="black", lw=1)
    ax[1].set_xticks(x); ax[1].set_xticklabels([f"g{int(g)}" for g in desac["grupo"]])
    ax[1].set_xlabel("quintil de desacuerdo"); ax[1].set_ylabel("skill sobre el mercado")
    ax[1].set_title("1 − RPS_modelo / RPS_mercado (positivo = aporta)")
    plt.tight_layout(); plt.show()

    extremo = desac.iloc[-1]
    print(f"en el quintil de mayor desacuerdo (distancia media "
          f"{extremo['dist_media']:.3f}, n={int(extremo['n'])}) el skill es "
          f"{extremo['skill_rps']:+.4f}")

## D.5 · La economía, emparejada con el listón

El apostante de referencia sigue siempre al favorito de Bet365 y respalda su cuota más
baja. Es un rival deliberadamente honesto: opera sobre los mismos partidos, con el
mismo importe y en el mismo orden temporal, de modo que la diferencia entre las dos
curvas no puede atribuirse a haber apostado a cosas distintas.

La curva de capital es ilustrativa, no probatoria: depende del orden en que ocurrieron
los partidos, y ese orden fue uno entre muchos posibles. La cifra que se defiende es el
**rendimiento sobre importe arriesgado** con su intervalo por remuestreo de jornadas.

In [ ]:
libros = E.economia["libros"]
iv = E.economia.get("intervalo") or {}

bt.imprime_comparacion(E.economia)
if iv:
    print(f"\nintervalo de la ventaja al 95 %: [{iv['inf']:+.4f}, {iv['sup']:+.4f}]")
    if iv["inf"] <= 0 <= iv["sup"]:
        print("el intervalo cruza el cero: la ventaja NO es distinguible de cero.")
    else:
        print("el intervalo no cruza el cero.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))

for etiqueta, color in (("modelo", PALETA["modelo"]), ("referencia", PALETA["mercado"])):
    curva = bt.equity_curve(libros[etiqueta])
    if not curva.empty:
        ax[0].plot(curva["date"], curva["banca"], lw=1.5, color=color, label=etiqueta)
ax[0].axhline(bt.BANCA_INICIAL, ls="--", lw=1, color="k")
ax[0].set_ylabel("banca"); ax[0].set_title("Trayectoria del banco")
ax[0].legend(fontsize=8)

pj = E.economia["por_jornada"]
if not pj.empty:
    ax[1].plot(pj["fecha"], pj["rendimiento_acum_modelo"], lw=1.5,
               color=PALETA["modelo"], label="modelo")
    ax[1].plot(pj["fecha"], pj["rendimiento_acum_ref"], lw=1.5,
               color=PALETA["mercado"], label="referencia")
    ax[1].axhline(0, ls="--", lw=1, color="k")
    ax[1].set_ylabel("rendimiento acumulado")
    ax[1].set_title("Rendimiento acumulado, jornada a jornada")
    ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

display(E.economia["por_temporada"].round(4))

In [ ]:
barrido_umbral = bt.sweep_umbral(pub)
boot = bt.bootstrap_rendimiento(pub, n_boot=400, seed=7)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(barrido_umbral["umbral"], barrido_umbral["rendimiento"], marker="o",
           color=PALETA["modelo"])
ax[0].axhline(0, ls="--", lw=1, color="k")
ax[0].axvline(bt.UMBRAL_VALOR, ls="--", lw=1.2, color=PALETA["apoyo"],
              label=f"umbral declarado ({bt.UMBRAL_VALOR})")
ax[0].set_xlabel("umbral de valor"); ax[0].set_ylabel("rendimiento")
ax[0].set_title("Forma de la curva de exigencia"); ax[0].legend(fontsize=8)
eje2 = ax[0].twinx()
eje2.bar(barrido_umbral["umbral"], barrido_umbral["n_apuestas"], width=.004,
         alpha=.25, color=PALETA["apoyo"])
eje2.set_ylabel("apuestas")

ivs = {"rendimiento del modelo": boot}
if E.economia.get("intervalo"):
    ivs["ventaja sobre la referencia"] = E.economia["intervalo"]
pos = np.arange(len(ivs))
for k, (etiqueta, d) in enumerate(ivs.items()):
    ax[1].errorbar(d["estimacion"], k,
                   xerr=[[d["estimacion"] - d["inf"]], [d["sup"] - d["estimacion"]]],
                   fmt="o", ms=7, capsize=5, color=PALETA["modelo"])
ax[1].axvline(0, ls="--", lw=1.2, color="k")
ax[1].set_yticks(pos); ax[1].set_yticklabels(list(ivs), fontsize=8)
ax[1].set_xlabel("valor")
ax[1].set_title(f"Intervalos al 95 % ({boot['n_replicas']} réplicas de jornadas)")
plt.tight_layout(); plt.show()

display(barrido_umbral.round(4))
print("la tabla no elige el umbral —eso sería ajustar la política sobre los mismos "
      "partidos que la juzgan—, sino comprobar que la curva tiene forma.")

## D.6 · Qué está mirando el modelo

La interpretabilidad se calcula sobre unos pocos pliegues reentrenados: los más
recientes, que son los que describen el modelo que se publicaría. `lb.ajusta` los rehace
reutilizando la caché de estados, de modo que no vuelve a ajustar ningún preprocesado.

Dos lecturas complementarias. La **importancia por permutación por bloques** sustituye
el bloque entero por filas sorteadas de la ventana de entrenamiento y mide cuánto
empeora el pronóstico: conserva la correlación interna del bloque en lugar de construir
partidos imposibles. Los **coeficientes** —sólo para la logística— dicen la dirección,
que la permutación no da.

In [ ]:
# ---- EDITABLE -------------------------------------------------------------
N_PLIEGUES_INTERPRETACION = 6
# ---------------------------------------------------------------------------

ajustados = lb.ajusta(E.columnas, E.familia, hiper=E.hiper, cfg=E.cfg, feat=feat,
                      n_pliegues=N_PLIEGUES_INTERPRETACION)
imp = ex.permutation_importance_blocks(ajustados, columnas=E.columnas,
                                       metrica="rps", n_repeticiones=5, feat=feat)
display(imp.round(5))

fig, ax = plt.subplots(figsize=(11, max(3, 0.42 * len(imp))))
d = imp.sort_values("importancia")
ax.barh(d["grupo"], d["importancia"], xerr=d["ee"], color=PALETA["modelo"])
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("deterioro del RPS al romper el bloque")
ax.set_title(f"Importancia por bloques · {E.nombre} · "
             f"{imp.attrs.get('n_pliegues')} pliegues")
plt.tight_layout(); plt.show()

In [ ]:
if E.familia == "logistica":
    coef = ajustados[-1].modelo.coeficientes()
    display(coef.round(4))
    try:
        display(ex.coefficients_by_block(coef).round(4))
    except KeyError as err:
        print(f"agregación por bloque no disponible: {err}")
else:
    print(f"la familia '{E.familia}' no expone coeficientes lineales; "
          "la lectura de dirección sólo aplica a la logística")

---
# Parte E · Cierre

Cuando el conjunto está elegido, quedan cuatro pasos: afinar los hiperparámetros,
declararlo en el código, abrir la reserva una sola vez y publicar.

## E.1 · Afinado corto sobre el conjunto ya elegido

Ahora —y sólo ahora— se mueven los hiperparámetros. El espacio es deliberadamente corto
(`lb.ESPACIOS_FINAL`, dos ejes por familia): con el conjunto fijo, un barrido amplio
compraría décimas de RPS a cambio de un riesgo de sobreajuste al propio esquema de
validación que no compensa.

El orden es por **RPS**, no por ventaja económica: el rendimiento sobre un puñado de
apuestas es demasiado ruidoso para arbitrar entre configuraciones parecidas.

In [ ]:
# ---- EDITABLE -------------------------------------------------------------
# Por defecto, lo que haya elegido la búsqueda de C.9; si no se ha ejecutado en esta
# sesión, el conjunto escrito a mano en la Parte C. Para retomar una búsqueda de otro
# día sin repetirla: CONJUNTO_FINAL = list(sl.carga()["conjunto"]).
CONJUNTO_FINAL = list(sel.conjunto) if "sel" in globals() else CONJUNTO
FAMILIA_FINAL  = FAMILIA_SEL if "sel" in globals() else FAMILIA
# ---------------------------------------------------------------------------
print(f"al afinado van {len(CONJUNTO_FINAL)} variables, familia {FAMILIA_FINAL}")

afinado = lb.afina(CONJUNTO_FINAL, FAMILIA_FINAL, feat=feat)
display(afinado)

## E.2 · El conjunto declarado en el código

La elección se cierra escribiéndola en `build_models.py`. A partir de ese momento el
conjunto deja de ser una variable del cuaderno y pasa a ser parte del artefacto: el
recorrido semanal, el bloque de reserva y la publicación leen de ahí.

El módulo expone seis constantes, y las seis se leen aquí para comprobar que el código
y el cuaderno dicen lo mismo:

| constante | qué es |
|---|---|
| `bm.CONJUNTO` | la tupla cerrada de variables |
| `bm.FAMILIA` | la familia elegida |
| `bm.HIPER` | su configuración, ya afinada |
| `bm.PROTOCOLO` | el `ProtocolConfig` con el que se midió |
| `bm.PROCEDENCIA` | el nombre del ensayo del registro del que salió |
| `bm.LIMITACIONES` | lo que el conjunto no cubre, dicho por escrito |

El conjunto **no se teclea a mano** en `build_models.py`. Si `bm.CONJUNTO` está vacío,
el módulo lo recoge de `seleccion.carga()` —el artefacto que la búsqueda dejó en
`outputs/models/seleccion/conjunto.json`— y comprueba antes que se eligió con el mismo
corte y la misma familia que rigen aquí; si no coinciden, lo ignora y lo dice. Copiar
la lista a mano crearía una segunda fuente de verdad, y las dos fuentes se separan el
día en que se repite la búsqueda y nadie se acuerda de actualizar la copia.

Escribir `CONJUNTO` explícitamente sigue valiendo, y tiene prioridad: es la vía para
cerrar un conjunto que no venga de la búsqueda.

### Y es lo que ocurre aquí: el conjunto publicado no sale de la búsqueda

Lo que `build_models.py` declara **no es** lo que eligió la voraz de C.9. Es la **mezcla
logit** de C.9.4: dos ajustes separados —uno sobre las tres columnas de precio, otro
sobre las 59 de juego— combinados como `p ∝ p_mercado^0.85 · p_juego^0.15` y
renormalizados, sobre las **62 columnas** que son la unión de las dos mitades. La
búsqueda sigue diciendo lo que midió —las tres variables de precio, en `conjunto.json`,
que este cuaderno **no toca**—, y el conjunto publicado la contiene: la mezcla añade las
59 de juego por una vía que no es la de añadir columnas a un ajuste.

**Por qué, y con qué autoridad.** Adoptarla exige **levantar la regla 4 del proyecto**
—cuando el ROI y el RPS discrepan, manda el RPS—, porque la mezcla **cuesta RPS**:
0.000527 peor que el modelo de solo precio al que sustituye, a cambio de +0.39 pp de
acierto y de un ROI plano que cambia de signo. La regla la levantó **Manuel el
19-oct-2026** —«quiero que el ROI mande, no el RPS»—, y **el alcance es la elección del
modelo publicado y nada más**: el RPS sigue rigiendo el piso frente a la previa de
frecuencias, la ordenación de la tabla comparativa y la importancia por permutación. No
es una decisión tomada en el código ni en este cuaderno.

**Por qué la mezcla y no cualquiera de las otras rutas.** De las **nueve** medidas en
C.9–C.9.4 buscando batir al precio, **ocho son nulas con el mismo patrón**: el signo se
invierte fuera del bloque que las eligió, que es la firma de haber escogido el máximo de
una colección. La mezcla es la única que no se invierte, y el motivo es estructural:
tiene **un solo parámetro** frente a los 2^65 conjuntos entre los que elegía la voraz,
de modo que apenas hay nada que sobreajustar.

**Y el peso se eligió por acierto, no por ROI**, que es el hallazgo de método de toda
esta parte y conviene no perderlo: elegir `w` por ROI sobre el bloque de selección da
w=0.35 y elegirlo por acierto da w=0.85, y fuera gana el segundo en las dos
convenciones de liquidación: **+2.51 % contra +1.38 %** a la mejor cuota entre casas y
**+1.08 % contra −0.05 %** de ventaja sobre el listón a cierre promediado. El ROI
sobre 1140 partidos tiene un error típico de unos 3 pp, así que
**seleccionar con él es seleccionar ruido**. La regla que queda escrita es la que
gobierna: **se selecciona por acierto y se reporta ROI.**

Por eso el control de sincronía de abajo no compara dos listas exigiendo igualdad. Tiene
que distinguir tres situaciones que no son la misma: que coincidan, que el conjunto
publicado **contenga** al de la búsqueda —el caso declarado, la mezcla— y que se hayan
separado de verdad, que es el único que denuncia un olvido.

In [ ]:
import build_models as bm

print(f"familia     : {bm.FAMILIA}")
print(f"hiper       : {bm.HIPER}")
print(f"protocolo   : {bm.PROTOCOLO.describe()}")
print(f"procedencia : {bm.PROCEDENCIA}")
print(f"variables   : {len(bm.CONJUNTO)}")
for c in bm.CONJUNTO:
    print("  ", c)
print("\nlimitaciones declaradas:")
for l in bm.LIMITACIONES:
    print("  ·", l)

In [ ]:
# El cuaderno y el codigo tienen que decir lo mismo, pero "lo mismo" aqui no es
# igualdad de listas: el conjunto publicado es la MEZCLA, y contiene al de la busqueda
# mas las 59 de juego que la mezcla incorpora por su segunda mitad. Son tres casos
# distintos y solo el tercero es una averia.
pub, cua = set(bm.CONJUNTO), set(CONJUNTO_FINAL)
falta_en_pub = sorted(cua - pub)
anade_pub = sorted(pub - cua)

if not falta_en_pub and not anade_pub:
    print(f"OK · el conjunto del cuaderno y el de build_models.py coinciden "
          f"({len(bm.CONJUNTO)} variables)")
elif not falta_en_pub:
    # Caso declarado: lo publicado CONTIENE lo que eligio la busqueda.
    print(f"OK · el publicado CONTIENE al del cuaderno: {len(cua)} -> "
          f"{len(pub)} variables, familia «{bm.FAMILIA}»")
    print(f"   anade {len(anade_pub)} columnas, que son la segunda mitad de la mezcla")
    print("   es la decision de C.9.4 y esta declarada: no es un desajuste")
else:
    print(f"DESAJUSTE · en el cuaderno y no en el publicado: {falta_en_pub}")
    print(f"   (el publicado anade, ademas: {anade_pub})")

try:
    procedente = lb.conjunto(bm.PROCEDENCIA)
    print(f"OK · '{bm.PROCEDENCIA}' está en el registro, con "
          f"{len(procedente)} variables")
except KeyError as err:
    print(f"AVISO · {err}")

## E.3 · La reserva, leída una sola vez

La temporada 2025/26 no se ha usado en ningún momento de este cuaderno. Se abre desde la
línea de órdenes, con el conjunto ya cerrado en el código:

```
python scripts/models/build_models.py --holdout
```

La cifra que salga de ahí es la que se publica en la memoria, y **no** se vuelve a
mirar el conjunto después. Si el resultado decepciona, la lectura honesta es que el
conjunto elegido no generalizaba: reabrir la búsqueda con ese dato en la mano
convertiría la reserva en un tercer conjunto de validación y dejaría al trabajo sin
ninguna estimación limpia.

In [ ]:
# Comprobación, no ejecución: el banco se niega a tocar la reserva.
try:
    lb.evalua(CONJUNTO_FINAL, FAMILIA_FINAL, nombre="_intento_holdout",
              evaluar="holdout", registrar=False, verbose=False, feat=feat)
    print("FALLO · el banco ha aceptado evaluar sobre la reserva")
except PermissionError as err:
    print("OK · el banco protege la reserva:")
    print("   ", str(err).split(".")[0])

## E.4 · Control de calidad y publicación

Cerrado el conjunto, el recorrido completo se ejecuta fuera del cuaderno:

```
python scripts/models/build_models.py          # entrena y evalúa el conjunto declarado
python scripts/models/qa_models.py             # el control de calidad de la fase
python scripts/models/publish_web.py           # los artefactos de la aplicación
```

`qa_models.py` comprueba sobre lo publicado lo que este cuaderno comprueba a mano —que
ninguna columna retrospectiva alimenta el modelo semanal, que las ternas de
probabilidad son distribuciones, que el holdout se abrió una sola vez, que modelo y
referencias están medidos sobre los mismos partidos y que ninguna cifra económica sale
sin la del apostante de Bet365 al lado— y falla el proceso si algo no se sostiene. Lo
que se comprueba aquí y no allí son propiedades de las funciones, no de los artefactos:
la identidad de Murphy, por ejemplo, la ejerce `test_evaluate.py`. El cuaderno explica;
el control de calidad es el que impide publicar.

## E.5 · Lo que este cuaderno no puede afirmar

Cinco límites que la memoria recoge y que conviene tener a la vista al leer cualquier
cifra de aquí:

1. **La búsqueda es voraz: encuentra un óptimo local, no el mejor conjunto posible.**
   El orden de entrada condiciona lo que entra después, y una variable inútil por sí
   sola —descartada en la criba— no vuelve a probarse aunque fuese útil acompañada. Lo
   que sí está resuelto es lo otro: se elige sobre las temporadas anteriores al corte y
   se publica sobre las posteriores, de modo que haber probado decenas de candidatas no
   infla la cifra citada. La estimación final sigue siendo la de la reserva.
2. **El rendimiento económico se simula sin fricción.** No hay límites de apuesta, ni
   movimiento del precio al entrar, ni comisiones, ni cuentas cerradas. Un rendimiento
   positivo aquí es una condición necesaria, no una demostración de rentabilidad.
3. **Las cuotas de cierre no son las que un apostante real habría conseguido.** Se usan
   porque son las más informativas y las que existen para todo el histórico, pero
   apostar al cierre exige estar en el mercado en ese momento exacto.
4. **El intervalo de la ventaja es amplio.** Con unas cuarenta jornadas por temporada,
   ninguna diferencia de rendimiento razonable es distinguible del azar dentro de una
   sola temporada. La conclusión, si la hay, descansa en el agregado y en la
   persistencia del signo, no en ninguna temporada concreta.
5. **El modelo no observa alineaciones confirmadas.** Trabaja con la información
   disponible antes del anuncio oficial, que es la situación de uso real, pero eso pone
   un techo a lo que puede saber de un partido concreto.